# 03 — Inspeção estrutural das saídas `middle.json`

Este notebook inspeciona a representação intermediária produzida pelo MinerU.

Objetivos:

- localizar os arquivos `*_middle.json` da amostra smoke;
- validar a estrutura geral dos documentos;
- inventariar tipos de blocos, linhas e spans;
- inspecionar títulos, tabelas, figuras e fórmulas;
- analisar cabeçalhos, rodapés e outros blocos descartados;
- identificar continuidade de conteúdo entre páginas;
- gerar tabelas consolidadas para orientar a construção do IR canônico.

Nenhuma normalização semântica será aplicada nesta etapa.

In [1]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path
from typing import Any, Iterable

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 220)

## 1. Caminhos do projeto

In [2]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


SMOKE_MANIFEST = PROJECT_ROOT / "data" / "samples" / "benchmark_smoke_sample.csv"

MINERU_OUTPUT_ROOT = PROJECT_ROOT / "artifacts" / "mineru" / "smoke"

REPORTS_ROOT = PROJECT_ROOT / "data" / "reports" / "middle_inspection"

REPORTS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


PROJECT_ROOT, SMOKE_MANIFEST, MINERU_OUTPUT_ROOT, REPORTS_ROOT

(WindowsPath('D:/baseia_v3'),
 WindowsPath('D:/baseia_v3/data/samples/benchmark_smoke_sample.csv'),
 WindowsPath('D:/baseia_v3/artifacts/mineru/smoke'),
 WindowsPath('D:/baseia_v3/data/reports/middle_inspection'))

## 2. Carregamento da amostra smoke

In [3]:
if not SMOKE_MANIFEST.exists():
    raise FileNotFoundError(f"Manifesto smoke não encontrado: {SMOKE_MANIFEST}")


smoke_sample = pd.read_csv(SMOKE_MANIFEST)

required_columns = {
    "filename",
    "path",
}

missing_columns = required_columns.difference(smoke_sample.columns)

if missing_columns:
    raise ValueError(f"Colunas ausentes no manifesto smoke: {sorted(missing_columns)}")


smoke_sample

,document_id,sha256,path,relative_path,filename,size_mb,page_count,encrypted,page_bucket,size_bucket,stratum
0,90d74e349676936f,90d74e349676936fe04ef15ec74301b7851f3bd20486492a0e569f1ae7e66f87,D:\baseia_v3\corpus\GDI_0685.pdf,GDI_0685.pdf,GDI_0685.pdf,1.5285,9.0,False,006_010,001_005mb,006_010|001_005mb|not_encrypted
1,1eda56df0d39f2ca,1eda56df0d39f2ca0eba1726db2215f38a6c239033145ab78f229df977bb7d88,D:\baseia_v3\corpus\GGH9.pdf,GGH9.pdf,GGH9.pdf,0.8927,8.0,False,006_010,025_001mb,006_010|025_001mb|not_encrypted
2,ce786e93290d48c3,ce786e93290d48c3297039cfcbe12238d1cdf71bb7563540f77a0b4c6e449c5f,D:\baseia_v3\corpus\GGT_1247.pdf,GGT_1247.pdf,GGT_1247.pdf,1.2042,10.0,False,006_010,001_005mb,006_010|001_005mb|not_encrypted
3,169ba37b912aa8dd,169ba37b912aa8ddf44315dce0edf8e7670219d53d515482c09005a680c03de7,D:\baseia_v3\corpus\GTL_0081.pdf,GTL_0081.pdf,GTL_0081.pdf,0.9256,5.0,False,003_005,025_001mb,003_005|025_001mb|not_encrypted
4,153ff9c09ca994ca,153ff9c09ca994ca35b79fede481f1340c0a29b5a969a0be401e6aac5330606b,D:\baseia_v3\corpus\GTM10.pdf,GTM10.pdf,GTM10.pdf,0.6307,9.0,False,006_010,025_001mb,006_010|025_001mb|not_encrypted


## 3. Localização dos arquivos `middle.json`

O MinerU normalmente cria um diretório por documento. A busca abaixo usa
primeiro o nome base do PDF e, em seguida, faz uma busca global pelo sufixo
`_middle.json`.

In [4]:
def normalize_stem(value: str | Path) -> str:
    return Path(str(value)).stem.casefold()


def discover_middle_files(
    output_root: Path,
) -> list[Path]:
    if not output_root.exists():
        return []

    return sorted(path for path in output_root.rglob("*_middle.json") if path.is_file())


all_middle_files = discover_middle_files(MINERU_OUTPUT_ROOT)

len(all_middle_files)

5

In [5]:
middle_files_by_stem: dict[str, list[Path]] = {}

for middle_path in all_middle_files:
    filename = middle_path.name

    if filename.endswith("_middle.json"):
        document_stem = filename.removesuffix("_middle.json")
    else:
        document_stem = middle_path.stem

    middle_files_by_stem.setdefault(
        document_stem.casefold(),
        [],
    ).append(middle_path)


def match_middle_file(
    filename: str,
    source_path: str | Path,
) -> tuple[Path | None, str]:
    candidate_stems = {
        normalize_stem(filename),
        normalize_stem(source_path),
    }

    exact_matches: list[Path] = []

    for candidate_stem in candidate_stems:
        exact_matches.extend(
            middle_files_by_stem.get(
                candidate_stem,
                [],
            )
        )

    exact_matches = sorted(set(exact_matches))

    if len(exact_matches) == 1:
        return exact_matches[0], "exact"

    if len(exact_matches) > 1:
        return exact_matches[0], "multiple_exact"

    fuzzy_matches = [
        path
        for path in all_middle_files
        if any(
            candidate_stem and candidate_stem in path.name.casefold()
            for candidate_stem in candidate_stems
        )
    ]

    if len(fuzzy_matches) == 1:
        return fuzzy_matches[0], "fuzzy"

    if len(fuzzy_matches) > 1:
        return fuzzy_matches[0], "multiple_fuzzy"

    return None, "not_found"


middle_manifest_rows: list[dict[str, Any]] = []

for row in smoke_sample.itertuples(index=False):
    middle_path, match_method = match_middle_file(
        filename=str(row.filename),
        source_path=str(row.path),
    )

    middle_manifest_rows.append(
        {
            "filename": row.filename,
            "source_path": row.path,
            "middle_path": (str(middle_path) if middle_path is not None else None),
            "middle_exists": (middle_path is not None and middle_path.exists()),
            "match_method": match_method,
        }
    )


middle_manifest = pd.DataFrame(middle_manifest_rows)

middle_manifest

,filename,source_path,middle_path,middle_exists,match_method
0,GDI_0685.pdf,D:\baseia_v3\corpus\GDI_0685.pdf,D:\baseia_v3\artifacts\mineru\smoke\90d74e349676936f\GDI_0685\auto\GDI_0685_middle.json,True,exact
1,GGH9.pdf,D:\baseia_v3\corpus\GGH9.pdf,D:\baseia_v3\artifacts\mineru\smoke\1eda56df0d39f2ca\GGH9\auto\GGH9_middle.json,True,exact
2,GGT_1247.pdf,D:\baseia_v3\corpus\GGT_1247.pdf,D:\baseia_v3\artifacts\mineru\smoke\ce786e93290d48c3\GGT_1247\auto\GGT_1247_middle.json,True,exact
3,GTL_0081.pdf,D:\baseia_v3\corpus\GTL_0081.pdf,D:\baseia_v3\artifacts\mineru\smoke\169ba37b912aa8dd\GTL_0081\auto\GTL_0081_middle.json,True,exact
4,GTM10.pdf,D:\baseia_v3\corpus\GTM10.pdf,D:\baseia_v3\artifacts\mineru\smoke\153ff9c09ca994ca\GTM10\auto\GTM10_middle.json,True,exact


In [6]:
missing_middle = middle_manifest[~middle_manifest["middle_exists"]].copy()

if not missing_middle.empty:
    display(Markdown("### Arquivos `middle.json` não encontrados"))
    display(missing_middle)
else:
    display(
        Markdown("Todos os arquivos `middle.json` da amostra smoke foram encontrados.")
    )

Todos os arquivos `middle.json` da amostra smoke foram encontrados.

## 4. Leitura e validação básica dos JSONs

In [7]:
def load_json(
    path: Path,
) -> Any:
    with path.open(
        "r",
        encoding="utf-8",
    ) as file:
        return json.load(file)


def validate_middle_document(
    data: Any,
) -> list[str]:
    errors: list[str] = []

    if not isinstance(data, dict):
        return ["A raiz do middle.json não é um objeto JSON."]

    pdf_info = data.get("pdf_info")

    if not isinstance(pdf_info, list):
        errors.append("`pdf_info` não existe ou não é uma lista.")
        return errors

    for page_index, page in enumerate(pdf_info):
        if not isinstance(page, dict):
            errors.append(f"Página {page_index}: valor não é objeto.")
            continue

        if "page_idx" not in page:
            errors.append(f"Página {page_index}: `page_idx` ausente.")

        for field in (
            "preproc_blocks",
            "para_blocks",
            "discarded_blocks",
        ):
            value = page.get(field)

            if value is not None and not isinstance(
                value,
                list,
            ):
                errors.append(f"Página {page_index}: `{field}` não é lista.")

    return errors


loaded_documents: dict[str, dict[str, Any]] = {}
validation_rows: list[dict[str, Any]] = []

for row in middle_manifest.itertuples(index=False):
    if not row.middle_exists:
        validation_rows.append(
            {
                "filename": row.filename,
                "middle_path": row.middle_path,
                "loaded": False,
                "valid": False,
                "page_count": None,
                "error_count": 1,
                "errors": "Arquivo não encontrado.",
            }
        )
        continue

    middle_path = Path(row.middle_path)

    try:
        data = load_json(middle_path)
        errors = validate_middle_document(data)

        document_id = normalize_stem(row.filename)

        loaded_documents[document_id] = {
            "filename": row.filename,
            "source_path": row.source_path,
            "middle_path": middle_path,
            "data": data,
        }

        validation_rows.append(
            {
                "filename": row.filename,
                "middle_path": str(middle_path),
                "loaded": True,
                "valid": not errors,
                "page_count": len(
                    data.get(
                        "pdf_info",
                        [],
                    )
                ),
                "error_count": len(errors),
                "errors": " | ".join(errors),
            }
        )

    except Exception as exc:
        validation_rows.append(
            {
                "filename": row.filename,
                "middle_path": str(middle_path),
                "loaded": False,
                "valid": False,
                "page_count": None,
                "error_count": 1,
                "errors": (f"{type(exc).__name__}: {exc}"),
            }
        )


validation_df = pd.DataFrame(validation_rows)

validation_df

,filename,middle_path,loaded,valid,page_count,error_count,errors
0,GDI_0685.pdf,D:\baseia_v3\artifacts\mineru\smoke\90d74e349676936f\GDI_0685\auto\GDI_0685_middle.json,True,True,9,0,
1,GGH9.pdf,D:\baseia_v3\artifacts\mineru\smoke\1eda56df0d39f2ca\GGH9\auto\GGH9_middle.json,True,True,8,0,
2,GGT_1247.pdf,D:\baseia_v3\artifacts\mineru\smoke\ce786e93290d48c3\GGT_1247\auto\GGT_1247_middle.json,True,True,10,0,
3,GTL_0081.pdf,D:\baseia_v3\artifacts\mineru\smoke\169ba37b912aa8dd\GTL_0081\auto\GTL_0081_middle.json,True,True,5,0,
4,GTM10.pdf,D:\baseia_v3\artifacts\mineru\smoke\153ff9c09ca994ca\GTM10\auto\GTM10_middle.json,True,True,9,0,


In [8]:
if not loaded_documents:
    raise RuntimeError("Nenhum middle.json pôde ser carregado.")

## 5. Funções auxiliares de navegação

In [9]:
def iter_pages(
    document: dict[str, Any],
) -> Iterable[tuple[int, dict[str, Any]]]:
    pdf_info = document.get(
        "pdf_info",
        [],
    )

    for fallback_index, page in enumerate(pdf_info):
        page_index = page.get(
            "page_idx",
            fallback_index,
        )

        yield int(page_index), page


def iter_blocks(
    document: dict[str, Any],
    field: str,
) -> Iterable[tuple[int, int, dict[str, Any]]]:
    for page_index, page in iter_pages(document):
        blocks = (
            page.get(
                field,
                [],
            )
            or []
        )

        for fallback_index, block in enumerate(blocks):
            block_index = block.get(
                "index",
                fallback_index,
            )

            yield (
                page_index,
                int(block_index),
                block,
            )


def iter_lines(
    block: dict[str, Any],
) -> Iterable[tuple[int, dict[str, Any]]]:
    lines = (
        block.get(
            "lines",
            [],
        )
        or []
    )

    for line_index, line in enumerate(lines):
        yield line_index, line


def iter_spans(
    line: dict[str, Any],
) -> Iterable[tuple[int, dict[str, Any]]]:
    spans = (
        line.get(
            "spans",
            [],
        )
        or []
    )

    for span_index, span in enumerate(spans):
        yield span_index, span


def extract_text_from_span(
    span: dict[str, Any],
) -> str:
    for key in (
        "content",
        "text",
        "latex",
        "html",
    ):
        value = span.get(key)

        if value not in (
            None,
            "",
        ):
            return str(value)

    return ""


def extract_text_from_line(
    line: dict[str, Any],
) -> str:
    texts = [extract_text_from_span(span) for _, span in iter_spans(line)]

    return " ".join(text for text in texts if text).strip()


def extract_text_from_block(
    block: dict[str, Any],
) -> str:
    direct_candidates = (
        block.get("content"),
        block.get("text"),
        block.get("latex"),
        block.get("html"),
    )

    for candidate in direct_candidates:
        if isinstance(candidate, str) and candidate.strip():
            return candidate.strip()

    lines = [extract_text_from_line(line) for _, line in iter_lines(block)]

    return " ".join(line for line in lines if line).strip()


def bbox_metrics(
    bbox: Any,
) -> dict[str, float | None]:
    if not isinstance(bbox, list) or len(bbox) != 4:
        return {
            "x0": None,
            "y0": None,
            "x1": None,
            "y1": None,
            "width": None,
            "height": None,
            "area": None,
        }

    try:
        x0, y0, x1, y1 = map(
            float,
            bbox,
        )
    except (
        TypeError,
        ValueError,
    ):
        return {
            "x0": None,
            "y0": None,
            "x1": None,
            "y1": None,
            "width": None,
            "height": None,
            "area": None,
        }

    width = max(
        0.0,
        x1 - x0,
    )
    height = max(
        0.0,
        y1 - y0,
    )

    return {
        "x0": x0,
        "y0": y0,
        "x1": x1,
        "y1": y1,
        "width": width,
        "height": height,
        "area": width * height,
    }


def truncate_text(
    value: Any,
    limit: int = 240,
) -> str:
    text = str(value or "").replace(
        "\n",
        " ",
    )

    text = " ".join(text.split())

    if len(text) <= limit:
        return text

    return text[: limit - 1] + "…"

## 6. Tabela plana de blocos

Cada linha representa um bloco do `middle.json`.

São mantidas separadamente as três coleções principais:

- `preproc_blocks`;
- `para_blocks`;
- `discarded_blocks`.

In [10]:
BLOCK_FIELDS = (
    "preproc_blocks",
    "para_blocks",
    "discarded_blocks",
)


block_rows: list[dict[str, Any]] = []

for document_id, document_record in loaded_documents.items():
    document = document_record["data"]

    for block_collection in BLOCK_FIELDS:
        for (
            page_index,
            block_index,
            block,
        ) in iter_blocks(
            document,
            block_collection,
        ):
            bbox = bbox_metrics(block.get("bbox"))

            lines = (
                block.get(
                    "lines",
                    [],
                )
                or []
            )

            spans = [span for line in lines for _, span in iter_spans(line)]

            block_rows.append(
                {
                    "document_id": document_id,
                    "filename": document_record["filename"],
                    "middle_path": str(document_record["middle_path"]),
                    "block_collection": (block_collection),
                    "page_index": page_index,
                    "page_number": page_index + 1,
                    "block_index": block_index,
                    "block_type": block.get("type"),
                    "level": block.get("level"),
                    "score": block.get("score"),
                    "line_count": len(lines),
                    "span_count": len(spans),
                    "text": extract_text_from_block(block),
                    "text_length": len(extract_text_from_block(block)),
                    "has_bbox": (block.get("bbox") is not None),
                    "has_bbox_fs": (block.get("bbox_fs") is not None),
                    "lines_deleted": bool(
                        block.get(
                            "lines_deleted",
                            False,
                        )
                    ),
                    "has_blocks": bool(block.get("blocks")),
                    "has_image_path": bool(
                        block.get("image_path") or block.get("img_path")
                    ),
                    "has_html": bool(block.get("html")),
                    "has_latex": bool(block.get("latex")),
                    **bbox,
                }
            )


blocks_df = pd.DataFrame(block_rows)

blocks_df.shape

(877, 28)

In [11]:
blocks_df.head(20)

,document_id,filename,middle_path,block_collection,page_index,page_number,block_index,block_type,level,score,line_count,span_count,text,text_length,has_bbox,has_bbox_fs,lines_deleted,has_blocks,has_image_path,has_html,has_latex,x0,y0,x1,y1,width,height,area
0,gdi_0685,GDI_0685.pdf,D:\baseia_v3\artifacts\mineru\smoke\90d74e349676936f\GDI_0685\auto\GDI_0685_middle.json,preproc_blocks,0,1,3,title,1.0,0.9462,1,1,GRUPO DE ESTUDO DE SISTEMAS DE DISTRIBUIÇÃO - GDI,49,True,False,False,False,False,False,False,164.0,136.0,425.0,149.0,261.0,13.0,3393.0
1,gdi_0685,GDI_0685.pdf,D:\baseia_v3\artifacts\mineru\smoke\90d74e349676936f\GDI_0685\auto\GDI_0685_middle.json,preproc_blocks,0,1,4,title,1.0,0.9652,2,2,DESENVOLVIMENTO DE CAMINHÃO ELÉTRICO PARA UTILIZAÇÃO EM MANUTENÇÃO EM REDES DE DISTRIBUIÇÃO DE BAIXA TENSÃO,107,True,False,False,False,False,False,False,74.0,167.0,515.0,190.0,441.0,23.0,10143.0
2,gdi_0685,GDI_0685.pdf,D:\baseia_v3\artifacts\mineru\smoke\90d74e349676936f\GDI_0685\auto\GDI_0685_middle.json,preproc_blocks,0,1,5,text,NaN,0.7053,5,5,ANA PAULA OENING (1); SIGNIE LAUREANO FRANÇA SANTOS (1); NATAN BARBIZAN (2); LEONARDO LOMBARDO FERREIRA (2); EDEMIR LUIZ KOWALSKI (1); GUILHERME RACHELLE HE...,305,True,False,False,False,False,False,False,69.0,209.0,516.0,263.0,447.0,54.0,24138.0
3,gdi_0685,GDI_0685.pdf,D:\baseia_v3\artifacts\mineru\smoke\90d74e349676936f\GDI_0685\auto\GDI_0685_middle.json,preproc_blocks,0,1,6,title,2.0,0.9274,1,1,RESUMO,6,True,False,False,False,False,False,False,65.0,281.0,109.0,293.0,44.0,12.0,528.0
4,gdi_0685,GDI_0685.pdf,D:\baseia_v3\artifacts\mineru\smoke\90d74e349676936f\GDI_0685\auto\GDI_0685_middle.json,preproc_blocks,0,1,7,abstract,NaN,0.9682,5,5,"Neste trabalho é apresentado o desenvolvimento de um caminhão elétrico para serviços de manutenção em redes de distribuição de energia elétrica, incluindo t...",511,True,False,False,False,False,False,False,64.0,298.0,533.0,352.0,469.0,54.0,25326.0
5,gdi_0685,GDI_0685.pdf,D:\baseia_v3\artifacts\mineru\smoke\90d74e349676936f\GDI_0685\auto\GDI_0685_middle.json,preproc_blocks,0,1,8,title,2.0,0.9336,1,1,PALAVRAS-CHAVE,14,True,False,False,False,False,False,False,65.0,366.0,150.0,378.0,85.0,12.0,1020.0
6,gdi_0685,GDI_0685.pdf,D:\baseia_v3\artifacts\mineru\smoke\90d74e349676936f\GDI_0685\auto\GDI_0685_middle.json,preproc_blocks,0,1,9,text,NaN,0.9438,1,1,Mobilidade elétrica; veículo elétrico; serviços de manutenção; recarga de oportunidade.,87,True,False,False,False,False,False,False,65.0,382.0,417.0,395.0,352.0,13.0,4576.0
7,gdi_0685,GDI_0685.pdf,D:\baseia_v3\artifacts\mineru\smoke\90d74e349676936f\GDI_0685\auto\GDI_0685_middle.json,preproc_blocks,0,1,10,title,2.0,0.9376,1,1,1.0 INTRODUÇÃO,14,True,False,False,False,False,False,False,66.0,403.0,145.0,415.0,79.0,12.0,948.0
8,gdi_0685,GDI_0685.pdf,D:\baseia_v3\artifacts\mineru\smoke\90d74e349676936f\GDI_0685\auto\GDI_0685_middle.json,preproc_blocks,0,1,11,text,NaN,0.9876,8,8,Em 2018 foi firmada uma parceria estratégica entre a ANEEL e a Agência de Cooperação Alemã (GIZ) para a organização da estruturação da Rede de Inovação do S...,839,True,False,False,False,False,False,False,64.0,420.0,533.0,505.0,469.0,85.0,39865.0
9,gdi_0685,GDI_0685.pdf,D:\baseia_v3\artifacts\mineru\smoke\90d74e349676936f\GDI_0685\auto\GDI_0685_middle.json,preproc_blocks,0,1,12,text,NaN,0.9868,6,6,"Neste sentido, com o intuito de contribuir para os objetivos propostos na temática da eletromobilidade no país e alcançar os resultados a que se propõe [2],...",648,True,False,False,False,False,False,False,64.0,509.0,533.0,573.0,469.0,64.0,30016.0


## 7. Inventário de tipos de bloco

In [12]:
block_type_counts = (
    blocks_df.groupby(
        [
            "block_collection",
            "block_type",
        ],
        dropna=False,
    )
    .size()
    .rename("count")
    .reset_index()
    .sort_values(
        [
            "block_collection",
            "count",
            "block_type",
        ],
        ascending=[
            True,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

block_type_counts

,block_collection,block_type,count
0,discarded_blocks,page_number,36
1,discarded_blocks,header,14
2,discarded_blocks,footer,11
3,para_blocks,text,215
4,para_blocks,title,69
5,para_blocks,ref_text,48
6,para_blocks,image,38
7,para_blocks,chart,20
8,para_blocks,table,10
9,para_blocks,interline_equation,4


In [13]:
block_type_by_document = blocks_df.pivot_table(
    index="filename",
    columns=[
        "block_collection",
        "block_type",
    ],
    values="block_index",
    aggfunc="count",
    fill_value=0,
).sort_index(axis=1)

block_type_by_document

block_collection discarded_blocks                    para_blocks                                                               preproc_blocks                                                         
block_type                 footer header page_number    abstract chart image interline_equation list ref_text table text title       abstract chart image interline_equation ref_text table text title
filename                                                                                                                                                                                              
GDI_0685.pdf                    1      2           8           1     0    17                  0    1        5     2   47    15              1     0    17                  0        5     2   48    15
GGH9.pdf                        1      3           7           1     5     5                  0    0        0     2   48    12              1     5     5                  0        0     2   48    12
GGT_1247.pdf                    2      3           9           1     8     1                  4    0       33     5   52    17              1     8     1                  4       33     5   52    17
GTL_0081.pdf                    6      2           4           0     0     8                  0    0        0     0   20    12              0     0     8                  0        0     0   20    12
GTM10.pdf                       1      4           8           0     7     7                  0    0       10     1   48    13              0     7     7                  0       10     1   48    13

## 8. Estrutura de páginas e coleções de blocos

In [14]:
page_rows: list[dict[str, Any]] = []

for document_id, document_record in loaded_documents.items():
    for page_index, page in iter_pages(document_record["data"]):
        page_size = page.get(
            "page_size",
            [],
        )

        page_width = (
            page_size[0]
            if isinstance(page_size, list) and len(page_size) >= 2
            else None
        )

        page_height = (
            page_size[1]
            if isinstance(page_size, list) and len(page_size) >= 2
            else None
        )

        page_rows.append(
            {
                "document_id": document_id,
                "filename": document_record["filename"],
                "page_index": page_index,
                "page_number": page_index + 1,
                "page_width": page_width,
                "page_height": page_height,
                "preproc_block_count": len(
                    page.get(
                        "preproc_blocks",
                        [],
                    )
                    or []
                ),
                "para_block_count": len(
                    page.get(
                        "para_blocks",
                        [],
                    )
                    or []
                ),
                "discarded_block_count": len(
                    page.get(
                        "discarded_blocks",
                        [],
                    )
                    or []
                ),
            }
        )


pages_df = pd.DataFrame(page_rows)

pages_df

,document_id,filename,page_index,page_number,page_width,page_height,preproc_block_count,para_block_count,discarded_block_count
0,gdi_0685,GDI_0685.pdf,0,1,595,841,14,14,3
1,gdi_0685,GDI_0685.pdf,1,2,595,841,5,5,1
2,gdi_0685,GDI_0685.pdf,2,3,595,841,10,10,1
3,gdi_0685,GDI_0685.pdf,3,4,595,841,14,14,1
4,gdi_0685,GDI_0685.pdf,4,5,595,841,7,7,1
5,gdi_0685,GDI_0685.pdf,5,6,595,841,11,11,1
6,gdi_0685,GDI_0685.pdf,6,7,595,841,12,12,1
7,gdi_0685,GDI_0685.pdf,7,8,595,841,5,5,1
8,gdi_0685,GDI_0685.pdf,8,9,595,841,10,10,1
9,ggh9,GGH9.pdf,0,1,595,842,13,13,4


In [15]:
page_summary = pages_df.groupby(
    "filename",
    as_index=False,
).agg(
    page_count=(
        "page_index",
        "nunique",
    ),
    preproc_blocks=(
        "preproc_block_count",
        "sum",
    ),
    para_blocks=(
        "para_block_count",
        "sum",
    ),
    discarded_blocks=(
        "discarded_block_count",
        "sum",
    ),
    min_page_width=(
        "page_width",
        "min",
    ),
    max_page_width=(
        "page_width",
        "max",
    ),
    min_page_height=(
        "page_height",
        "min",
    ),
    max_page_height=(
        "page_height",
        "max",
    ),
)

page_summary

,filename,page_count,preproc_blocks,para_blocks,discarded_blocks,min_page_width,max_page_width,min_page_height,max_page_height
0,GDI_0685.pdf,9,88,88,11,595,595,841,841
1,GGH9.pdf,8,73,73,11,595,595,842,842
2,GGT_1247.pdf,10,121,121,14,595,595,841,841
3,GTL_0081.pdf,5,40,40,12,595,595,841,841
4,GTM10.pdf,9,86,86,13,595,595,842,842


## 9. Níveis de título

In [16]:
title_blocks = blocks_df[
    blocks_df["block_type"].isin(
        {
            "title",
            "paragraph_title",
            "doc_title",
            "section_title",
        }
    )
    | blocks_df["level"].notna()
].copy()

title_blocks[
    [
        "filename",
        "block_collection",
        "page_number",
        "block_index",
        "block_type",
        "level",
        "score",
        "text",
    ]
].sort_values(
    [
        "filename",
        "page_number",
        "block_index",
        "block_collection",
    ]
)

,filename,block_collection,page_number,block_index,block_type,level,score,text
88,GDI_0685.pdf,para_blocks,1,3,title,1.0,0.9462,GRUPO DE ESTUDO DE SISTEMAS DE DISTRIBUIÇÃO - GDI
0,GDI_0685.pdf,preproc_blocks,1,3,title,1.0,0.9462,GRUPO DE ESTUDO DE SISTEMAS DE DISTRIBUIÇÃO - GDI
89,GDI_0685.pdf,para_blocks,1,4,title,1.0,0.9652,DESENVOLVIMENTO DE CAMINHÃO ELÉTRICO PARA UTILIZAÇÃO EM MANUTENÇÃO EM REDES DE DISTRIBUIÇÃO DE BAIXA TENSÃO
1,GDI_0685.pdf,preproc_blocks,1,4,title,1.0,0.9652,DESENVOLVIMENTO DE CAMINHÃO ELÉTRICO PARA UTILIZAÇÃO EM MANUTENÇÃO EM REDES DE DISTRIBUIÇÃO DE BAIXA TENSÃO
91,GDI_0685.pdf,para_blocks,1,6,title,2.0,0.9274,RESUMO
...,...,...,...,...,...,...,...,...
748,GTM10.pdf,preproc_blocks,7,4,title,2.0,0.9384,6.0 - CONCLUSÕES
842,GTM10.pdf,para_blocks,8,3,title,2.0,0.9396,7.0 - REFERÊNCIAS BIBLIOGRÁFICAS
756,GTM10.pdf,preproc_blocks,8,3,title,2.0,0.9396,7.0 - REFERÊNCIAS BIBLIOGRÁFICAS
853,GTM10.pdf,para_blocks,9,2,title,2.0,0.7603,8.0 - DADOS BIOGRÁFICOS


In [17]:
title_level_counts = (
    title_blocks.groupby(
        [
            "block_collection",
            "block_type",
            "level",
        ],
        dropna=False,
    )
    .size()
    .rename("count")
    .reset_index()
    .sort_values(
        [
            "block_collection",
            "block_type",
            "level",
        ]
    )
)

title_level_counts

,block_collection,block_type,level,count
0,para_blocks,title,1.0,12
1,para_blocks,title,2.0,57
2,preproc_blocks,title,1.0,12
3,preproc_blocks,title,2.0,57


## 10. Linhas e spans

Esta tabela preserva a granularidade necessária para diferenciar elementos
visualmente próximos dentro do mesmo bloco, como autores e filiação.

In [18]:
line_rows: list[dict[str, Any]] = []
span_rows: list[dict[str, Any]] = []

for document_id, document_record in loaded_documents.items():
    document = document_record["data"]

    for block_collection in BLOCK_FIELDS:
        for (
            page_index,
            block_index,
            block,
        ) in iter_blocks(
            document,
            block_collection,
        ):
            for line_index, line in iter_lines(block):
                line_bbox = bbox_metrics(line.get("bbox"))
                line_text = extract_text_from_line(line)

                spans = list(iter_spans(line))

                line_rows.append(
                    {
                        "document_id": document_id,
                        "filename": (document_record["filename"]),
                        "block_collection": (block_collection),
                        "page_index": page_index,
                        "page_number": (page_index + 1),
                        "block_index": block_index,
                        "block_type": block.get("type"),
                        "block_level": block.get("level"),
                        "line_index": line_index,
                        "line_text": line_text,
                        "line_text_length": len(line_text),
                        "span_count": len(spans),
                        **{f"line_{key}": value for key, value in line_bbox.items()},
                    }
                )

                for (
                    span_index,
                    span,
                ) in spans:
                    span_bbox = bbox_metrics(span.get("bbox"))

                    span_text = extract_text_from_span(span)

                    span_rows.append(
                        {
                            "document_id": (document_id),
                            "filename": (document_record["filename"]),
                            "block_collection": (block_collection),
                            "page_index": (page_index),
                            "page_number": (page_index + 1),
                            "block_index": (block_index),
                            "block_type": (block.get("type")),
                            "block_level": (block.get("level")),
                            "line_index": (line_index),
                            "span_index": (span_index),
                            "span_type": (span.get("type")),
                            "span_text": (span_text),
                            "span_text_length": (len(span_text)),
                            "score": (span.get("score")),
                            "cross_page": bool(
                                span.get(
                                    "cross_page",
                                    False,
                                )
                            ),
                            "has_latex": bool(span.get("latex")),
                            "has_html": bool(span.get("html")),
                            "has_image_path": bool(
                                span.get("image_path") or span.get("img_path")
                            ),
                            **{
                                f"span_{key}": (value)
                                for key, value in span_bbox.items()
                            },
                        }
                    )


lines_df = pd.DataFrame(line_rows)

spans_df = pd.DataFrame(span_rows)


lines_df.shape, spans_df.shape

((2475, 19), (2627, 25))

In [19]:
lines_df.head(20)

,document_id,filename,block_collection,page_index,page_number,block_index,block_type,block_level,line_index,line_text,line_text_length,span_count,line_x0,line_y0,line_x1,line_y1,line_width,line_height,line_area
0,gdi_0685,GDI_0685.pdf,preproc_blocks,0,1,3,title,1.0,0,GRUPO DE ESTUDO DE SISTEMAS DE DISTRIBUIÇÃO - GDI,49,1,165.0,137.0,424.0,149.0,259.0,12.0,3108.0
1,gdi_0685,GDI_0685.pdf,preproc_blocks,0,1,4,title,1.0,0,DESENVOLVIMENTO DE CAMINHÃO ELÉTRICO PARA UTILIZAÇÃO EM MANUTENÇÃO EM REDES DE,78,1,75.0,168.0,514.0,179.0,439.0,11.0,4829.0
2,gdi_0685,GDI_0685.pdf,preproc_blocks,0,1,4,title,1.0,1,DISTRIBUIÇÃO DE BAIXA TENSÃO,28,1,218.0,178.0,370.0,189.0,152.0,11.0,1672.0
3,gdi_0685,GDI_0685.pdf,preproc_blocks,0,1,5,text,NaN,0,ANA PAULA OENING (1); SIGNIE LAUREANO FRANÇA SANTOS (1); NATAN BARBIZAN (2); LEONARDO,85,1,71.0,212.0,516.0,223.0,445.0,11.0,4895.0
4,gdi_0685,GDI_0685.pdf,preproc_blocks,0,1,5,text,NaN,1,LOMBARDO FERREIRA (2); EDEMIR LUIZ KOWALSKI (1); GUILHERME RACHELLE HERNASKI (1);,81,1,81.0,220.0,505.0,234.0,424.0,14.0,5936.0
5,gdi_0685,GDI_0685.pdf,preproc_blocks,0,1,5,text,NaN,2,EDUARDO CASARIN (3); AMANDA CORTEZ (3),38,1,191.0,231.0,394.0,243.0,203.0,12.0,2436.0
6,gdi_0685,GDI_0685.pdf,preproc_blocks,0,1,5,text,NaN,3,INSTITUTO DE TECNOLOGIA PARA O DESENVOLVIMENTO (1); BYD DO BRASIL LTDA (2); ELEKTRO,83,1,74.0,241.0,512.0,255.0,438.0,14.0,6132.0
7,gdi_0685,GDI_0685.pdf,preproc_blocks,0,1,5,text,NaN,4,REDES S.A. (3),14,1,260.0,253.0,327.0,264.0,67.0,11.0,737.0
8,gdi_0685,GDI_0685.pdf,preproc_blocks,0,1,6,title,2.0,0,RESUMO,6,1,67.0,282.0,107.0,294.0,40.0,12.0,480.0
9,gdi_0685,GDI_0685.pdf,preproc_blocks,0,1,7,abstract,NaN,0,Neste trabalho é apresentado o desenvolvimento de um caminhão elétrico para serviços de manutenção em redes,107,1,66.0,299.0,531.0,308.0,465.0,9.0,4185.0


In [20]:
spans_df.head(20)

,document_id,filename,block_collection,page_index,page_number,block_index,block_type,block_level,line_index,span_index,span_type,span_text,span_text_length,score,cross_page,has_latex,has_html,has_image_path,span_x0,span_y0,span_x1,span_y1,span_width,span_height,span_area
0,gdi_0685,GDI_0685.pdf,preproc_blocks,0,1,3,title,1.0,0,0,text,GRUPO DE ESTUDO DE SISTEMAS DE DISTRIBUIÇÃO - GDI,49,1.0,False,False,False,False,165.0,137.0,424.0,149.0,259.0,12.0,3108.0
1,gdi_0685,GDI_0685.pdf,preproc_blocks,0,1,4,title,1.0,0,0,text,DESENVOLVIMENTO DE CAMINHÃO ELÉTRICO PARA UTILIZAÇÃO EM MANUTENÇÃO EM REDES DE,78,1.0,False,False,False,False,75.0,168.0,514.0,179.0,439.0,11.0,4829.0
2,gdi_0685,GDI_0685.pdf,preproc_blocks,0,1,4,title,1.0,1,0,text,DISTRIBUIÇÃO DE BAIXA TENSÃO,28,1.0,False,False,False,False,218.0,178.0,370.0,189.0,152.0,11.0,1672.0
3,gdi_0685,GDI_0685.pdf,preproc_blocks,0,1,5,text,NaN,0,0,text,ANA PAULA OENING (1); SIGNIE LAUREANO FRANÇA SANTOS (1); NATAN BARBIZAN (2); LEONARDO,85,1.0,False,False,False,False,71.0,212.0,516.0,223.0,445.0,11.0,4895.0
4,gdi_0685,GDI_0685.pdf,preproc_blocks,0,1,5,text,NaN,1,0,text,LOMBARDO FERREIRA (2); EDEMIR LUIZ KOWALSKI (1); GUILHERME RACHELLE HERNASKI (1);,81,1.0,False,False,False,False,81.0,220.0,505.0,234.0,424.0,14.0,5936.0
5,gdi_0685,GDI_0685.pdf,preproc_blocks,0,1,5,text,NaN,2,0,text,EDUARDO CASARIN (3); AMANDA CORTEZ (3),38,1.0,False,False,False,False,191.0,231.0,394.0,243.0,203.0,12.0,2436.0
6,gdi_0685,GDI_0685.pdf,preproc_blocks,0,1,5,text,NaN,3,0,text,INSTITUTO DE TECNOLOGIA PARA O DESENVOLVIMENTO (1); BYD DO BRASIL LTDA (2); ELEKTRO,83,1.0,False,False,False,False,74.0,241.0,512.0,255.0,438.0,14.0,6132.0
7,gdi_0685,GDI_0685.pdf,preproc_blocks,0,1,5,text,NaN,4,0,text,REDES S.A. (3),14,1.0,False,False,False,False,260.0,253.0,327.0,264.0,67.0,11.0,737.0
8,gdi_0685,GDI_0685.pdf,preproc_blocks,0,1,6,title,2.0,0,0,text,RESUMO,6,1.0,False,False,False,False,67.0,282.0,107.0,294.0,40.0,12.0,480.0
9,gdi_0685,GDI_0685.pdf,preproc_blocks,0,1,7,abstract,NaN,0,0,text,Neste trabalho é apresentado o desenvolvimento de um caminhão elétrico para serviços de manutenção em redes,107,1.0,False,False,False,False,66.0,299.0,531.0,308.0,465.0,9.0,4185.0


## 11. Tipos de span

In [21]:
if spans_df.empty:
    span_type_counts = pd.DataFrame(
        columns=[
            "span_type",
            "count",
        ]
    )
else:
    span_type_counts = (
        spans_df.groupby(
            "span_type",
            dropna=False,
        )
        .size()
        .rename("count")
        .reset_index()
        .sort_values(
            "count",
            ascending=False,
        )
        .reset_index(drop=True)
    )

span_type_counts

,span_type,count
0,text,2585
1,inline_equation,34
2,interline_equation,8


## 12. Blocos com múltiplas linhas

Estes blocos são especialmente importantes para a futura identificação de:

- autores;
- filiações;
- endereços;
- listas;
- títulos quebrados em mais de uma linha.

In [22]:
multi_line_blocks = (
    blocks_df[blocks_df["line_count"] > 1][
        [
            "filename",
            "block_collection",
            "page_number",
            "block_index",
            "block_type",
            "level",
            "line_count",
            "span_count",
            "score",
            "text",
        ]
    ]
    .sort_values(
        [
            "filename",
            "page_number",
            "block_index",
            "block_collection",
        ]
    )
    .reset_index(drop=True)
)

multi_line_blocks.head(100)

,filename,block_collection,page_number,block_index,block_type,level,line_count,span_count,score,text
0,GDI_0685.pdf,discarded_blocks,1,1,header,NaN,4,4,0.9276,XXVII Seminário Nacional de Produção e Transmissão de Energia Elétrica XXVIISNPTEE
1,GDI_0685.pdf,discarded_blocks,1,2,header,NaN,2,2,0.8573,0685 GDI/7
2,GDI_0685.pdf,para_blocks,1,4,title,1.0,2,2,0.9652,DESENVOLVIMENTO DE CAMINHÃO ELÉTRICO PARA UTILIZAÇÃO EM MANUTENÇÃO EM REDES DE DISTRIBUIÇÃO DE BAIXA TENSÃO
3,GDI_0685.pdf,preproc_blocks,1,4,title,1.0,2,2,0.9652,DESENVOLVIMENTO DE CAMINHÃO ELÉTRICO PARA UTILIZAÇÃO EM MANUTENÇÃO EM REDES DE DISTRIBUIÇÃO DE BAIXA TENSÃO
4,GDI_0685.pdf,para_blocks,1,5,list,NaN,5,5,0.7053,ANA PAULA OENING (1); SIGNIE LAUREANO FRANÇA SANTOS (1); NATAN BARBIZAN (2); LEONARDO LOMBARDO FERREIRA (2); EDEMIR LUIZ KOWALSKI (1); GUILHERME RACHELLE HE...
...,...,...,...,...,...,...,...,...,...,...
95,GGH9.pdf,discarded_blocks,1,2,header,NaN,5,5,0.9728,XXIII SNPTEE SEMINÁRIO NACIONAL DE PRODUÇÃO E TRANSMISSÃO DE ENERGIA ELÉTRICA
96,GGH9.pdf,discarded_blocks,1,3,header,NaN,3,3,0.9659,FI/GGH/09 18 a 21 de Outubro de 2015 Foz do Iguaçu - PR
97,GGH9.pdf,para_blocks,1,7,text,NaN,2,2,0.9657,Cornelis J. V. D. Poel Filho (*) ALSTOM ENERGIAS RENOVÁVEIS LTDA
98,GGH9.pdf,preproc_blocks,1,7,text,NaN,2,2,0.9657,Cornelis J. V. D. Poel Filho (*) ALSTOM ENERGIAS RENOVÁVEIS LTDA


In [23]:
multi_line_detail = (
    lines_df.merge(
        multi_line_blocks[
            [
                "filename",
                "block_collection",
                "page_number",
                "block_index",
            ]
        ].drop_duplicates(),
        on=[
            "filename",
            "block_collection",
            "page_number",
            "block_index",
        ],
        how="inner",
    )[
        [
            "filename",
            "block_collection",
            "page_number",
            "block_index",
            "block_type",
            "block_level",
            "line_index",
            "line_text",
            "line_x0",
            "line_y0",
            "line_x1",
            "line_y1",
        ]
    ]
    .sort_values(
        [
            "filename",
            "page_number",
            "block_index",
            "line_index",
            "block_collection",
        ]
    )
    .reset_index(drop=True)
)

multi_line_detail.head(200)

,filename,block_collection,page_number,block_index,block_type,block_level,line_index,line_text,line_x0,line_y0,line_x1,line_y1
0,GDI_0685.pdf,discarded_blocks,1,1,header,NaN,0,XXVII Seminário Nacional de,124.0,35.0,229.0,43.0
1,GDI_0685.pdf,discarded_blocks,1,1,header,NaN,1,Produção e Transmissão,124.0,45.0,248.0,55.0
2,GDI_0685.pdf,discarded_blocks,1,1,header,NaN,2,de Energia Elétrica,124.0,56.0,219.0,67.0
3,GDI_0685.pdf,discarded_blocks,1,1,header,NaN,3,XXVIISNPTEE,68.0,66.0,119.0,74.0
4,GDI_0685.pdf,discarded_blocks,1,2,header,NaN,0,0685,501.0,41.0,524.0,52.0
...,...,...,...,...,...,...,...,...,...,...,...,...
195,GDI_0685.pdf,preproc_blocks,4,6,text,NaN,2,procedimentos estabelecidos pela fabricante no que tange às características convencionais do caminhão de,66.0,324.0,531.0,333.0
196,GDI_0685.pdf,para_blocks,4,6,text,NaN,3,“mercado” vendido. Quanto às características específicas para o atendimento às necessidades do projeto de PD,68.0,334.0,531.0,343.0
197,GDI_0685.pdf,preproc_blocks,4,6,text,NaN,3,“mercado” vendido. Quanto às características específicas para o atendimento às necessidades do projeto de PD,68.0,334.0,531.0,343.0
198,GDI_0685.pdf,para_blocks,4,6,text,NaN,4,foram conduzidas de forma particular e em acordo com os requisitos apresentados na seção 2.1 deste trabalho. As,66.0,345.0,530.0,353.0


## 13. Inspeção da primeira página

A primeira página concentra frequentemente:

- grupo ou evento;
- título principal;
- autores;
- filiação;
- resumo;
- palavras-chave.

In [24]:
first_page_blocks = (
    blocks_df[blocks_df["page_index"] == 0][
        [
            "filename",
            "block_collection",
            "block_index",
            "block_type",
            "level",
            "score",
            "line_count",
            "span_count",
            "x0",
            "y0",
            "x1",
            "y1",
            "text",
        ]
    ]
    .sort_values(
        [
            "filename",
            "block_index",
            "block_collection",
        ]
    )
    .reset_index(drop=True)
)

first_page_blocks

,filename,block_collection,block_index,block_type,level,score,line_count,span_count,x0,y0,x1,y1,text
0,GDI_0685.pdf,discarded_blocks,1,header,NaN,0.9276,4,4,66.0,34.0,250.0,76.0,XXVII Seminário Nacional de Produção e Transmissão de Energia Elétrica XXVIISNPTEE
1,GDI_0685.pdf,discarded_blocks,2,header,NaN,0.8573,2,2,497.0,39.0,525.0,67.0,0685 GDI/7
2,GDI_0685.pdf,para_blocks,3,title,1.0,0.9462,1,1,164.0,136.0,425.0,149.0,GRUPO DE ESTUDO DE SISTEMAS DE DISTRIBUIÇÃO - GDI
3,GDI_0685.pdf,preproc_blocks,3,title,1.0,0.9462,1,1,164.0,136.0,425.0,149.0,GRUPO DE ESTUDO DE SISTEMAS DE DISTRIBUIÇÃO - GDI
4,GDI_0685.pdf,para_blocks,4,title,1.0,0.9652,2,2,74.0,167.0,515.0,190.0,DESENVOLVIMENTO DE CAMINHÃO ELÉTRICO PARA UTILIZAÇÃO EM MANUTENÇÃO EM REDES DE DISTRIBUIÇÃO DE BAIXA TENSÃO
...,...,...,...,...,...,...,...,...,...,...,...,...,...
165,GTM10.pdf,para_blocks,23,text,NaN,0.9405,1,1,83.0,655.0,264.0,666.0,Influência do fator de Hot-Spot variável;
166,GTM10.pdf,preproc_blocks,23,text,NaN,0.9405,1,1,83.0,655.0,264.0,666.0,Influência do fator de Hot-Spot variável;
167,GTM10.pdf,para_blocks,24,text,NaN,0.9693,2,2,64.0,671.0,532.0,694.0,"Em muitas análises ao longo deste trabalho será utilizado o ciclo denominado como “Ciclo de sobrecarga” da nota técnica elaborada pelo ONS [2], exibido na F..."
168,GTM10.pdf,preproc_blocks,24,text,NaN,0.9693,2,2,64.0,671.0,532.0,694.0,"Em muitas análises ao longo deste trabalho será utilizado o ciclo denominado como “Ciclo de sobrecarga” da nota técnica elaborada pelo ONS [2], exibido na F..."


In [25]:
first_page_lines = (
    lines_df[lines_df["page_index"] == 0][
        [
            "filename",
            "block_collection",
            "block_index",
            "block_type",
            "block_level",
            "line_index",
            "line_text",
            "line_x0",
            "line_y0",
            "line_x1",
            "line_y1",
        ]
    ]
    .sort_values(
        [
            "filename",
            "block_index",
            "line_index",
            "block_collection",
        ]
    )
    .reset_index(drop=True)
)

first_page_lines

,filename,block_collection,block_index,block_type,block_level,line_index,line_text,line_x0,line_y0,line_x1,line_y1
0,GDI_0685.pdf,discarded_blocks,1,header,NaN,0,XXVII Seminário Nacional de,124.0,35.0,229.0,43.0
1,GDI_0685.pdf,discarded_blocks,1,header,NaN,1,Produção e Transmissão,124.0,45.0,248.0,55.0
2,GDI_0685.pdf,discarded_blocks,1,header,NaN,2,de Energia Elétrica,124.0,56.0,219.0,67.0
3,GDI_0685.pdf,discarded_blocks,1,header,NaN,3,XXVIISNPTEE,68.0,66.0,119.0,74.0
4,GDI_0685.pdf,discarded_blocks,2,header,NaN,0,0685,501.0,41.0,524.0,52.0
...,...,...,...,...,...,...,...,...,...,...,...
443,GTM10.pdf,preproc_blocks,24,text,NaN,0,Em muitas análises ao longo deste trabalho será utilizado o ciclo denominado como “Ciclo de sobrecarga” da,83.0,672.0,530.0,683.0
444,GTM10.pdf,para_blocks,24,text,NaN,1,"nota técnica elaborada pelo ONS [2], exibido na Figura 1.",66.0,682.0,298.0,693.0
445,GTM10.pdf,preproc_blocks,24,text,NaN,1,"nota técnica elaborada pelo ONS [2], exibido na Figura 1.",66.0,682.0,298.0,693.0
446,GTM10.pdf,discarded_blocks,25,footer,NaN,0,"Rua Dr. Pedro Zimmermann, n˚ 6751 – CEP 89068-001 Blumenau, SC – Brasil",145.0,768.0,461.0,777.0


## 14. Tabelas

In [26]:
def find_nested_values(
    value: Any,
    target_keys: set[str],
) -> list[tuple[str, Any]]:
    matches: list[tuple[str, Any]] = []

    def walk(
        current: Any,
        path: str,
    ) -> None:
        if isinstance(current, dict):
            for key, child in current.items():
                child_path = f"{path}.{key}" if path else key

                if key in target_keys:
                    matches.append(
                        (
                            child_path,
                            child,
                        )
                    )

                walk(
                    child,
                    child_path,
                )

        elif isinstance(current, list):
            for index, child in enumerate(current):
                child_path = f"{path}[{index}]"

                walk(
                    child,
                    child_path,
                )

    walk(
        value,
        "",
    )

    return matches


table_rows: list[dict[str, Any]] = []

for document_id, document_record in loaded_documents.items():
    document = document_record["data"]

    for block_collection in BLOCK_FIELDS:
        for (
            page_index,
            block_index,
            block,
        ) in iter_blocks(
            document,
            block_collection,
        ):
            nested_html = find_nested_values(
                block,
                {"html"},
            )

            is_table = block.get("type") == "table" or bool(nested_html)

            if not is_table:
                continue

            html_values = [
                str(value)
                for _, value in nested_html
                if isinstance(
                    value,
                    str,
                )
                and value.strip()
            ]

            caption_values = [
                value
                for _, value in find_nested_values(
                    block,
                    {
                        "table_caption",
                        "caption",
                    },
                )
                if value
                not in (
                    None,
                    "",
                    [],
                )
            ]

            image_values = [
                value
                for _, value in find_nested_values(
                    block,
                    {
                        "image_path",
                        "img_path",
                    },
                )
                if value
            ]

            table_rows.append(
                {
                    "document_id": document_id,
                    "filename": document_record["filename"],
                    "block_collection": (block_collection),
                    "page_index": page_index,
                    "page_number": (page_index + 1),
                    "block_index": block_index,
                    "block_type": block.get("type"),
                    "score": block.get("score"),
                    "caption": truncate_text(caption_values),
                    "html": (html_values[0] if html_values else ""),
                    "html_length": sum(len(value) for value in html_values),
                    "image_paths": (
                        " | ".join(
                            map(
                                str,
                                image_values,
                            )
                        )
                    ),
                    "bbox": block.get("bbox"),
                }
            )


tables_df = pd.DataFrame(table_rows)

tables_df

,document_id,filename,block_collection,page_index,page_number,block_index,block_type,score,caption,html,html_length,image_paths,bbox
0,gdi_0685,GDI_0685.pdf,preproc_blocks,1,2,9,table,0.9851,,<table><tr><td rowspan=1 colspan=1>Distribuiçãode carga</td><td rowspan=1 colspan=1>VeículoCompleto(kg)</td><td rowspan=1 colspan=1>Peso BrutoTotal (kg)</td...,665,31c09c6d92f6710149776c78dce50432ad53ea940064bf1b4d4864a9f5c4bbbc.jpg,"[194, 681, 403, 754]"
1,gdi_0685,GDI_0685.pdf,preproc_blocks,6,7,3,table,0.9877,,<table><tr><td></td><td>Limeira</td><td>Mairiporã</td></tr><tr><td>Número de dias em movimento</td><td>32</td><td>40</td></tr><tr><td>Distância total percor...,864,a5eb75c41200c35f68490963e723cae78c38c6d21aa0c2bb717b2b7d4c5b519a.jpg,"[93, 98, 504, 215]"
2,gdi_0685,GDI_0685.pdf,para_blocks,1,2,9,table,0.9851,,<table><tr><td rowspan=1 colspan=1>Distribuiçãode carga</td><td rowspan=1 colspan=1>VeículoCompleto(kg)</td><td rowspan=1 colspan=1>Peso BrutoTotal (kg)</td...,665,31c09c6d92f6710149776c78dce50432ad53ea940064bf1b4d4864a9f5c4bbbc.jpg,"[194, 681, 403, 754]"
3,gdi_0685,GDI_0685.pdf,para_blocks,6,7,3,table,0.9877,,<table><tr><td></td><td>Limeira</td><td>Mairiporã</td></tr><tr><td>Número de dias em movimento</td><td>32</td><td>40</td></tr><tr><td>Distância total percor...,864,a5eb75c41200c35f68490963e723cae78c38c6d21aa0c2bb717b2b7d4c5b519a.jpg,"[93, 98, 504, 215]"
4,ggh9,GGH9.pdf,preproc_blocks,5,6,8,table,0.9877,,<table><tr><td rowspan=1 colspan=1>ESCOPO“b”</td><td rowspan=1 colspan=1>ESCOPO “C”APÓS AVALIAÇÃO DACONDIÇÃO E &quot;Pit Stop&quot;</td></tr><tr><td rowspan...,927,20545a34cec47b78d597a4ea5dc7323871094f252d694590eca2045513608c61.jpg,"[109, 554, 488, 657]"
5,ggh9,GGH9.pdf,preproc_blocks,6,7,8,table,0.9884,,<table><tr><td>PLANTA</td><td>ESCOPO</td><td>TEMPO DA PARADA SEM &quot;Pit Stop”</td><td>TEMPO DA PARADA COM “Pit Stop&quot;</td><td>ILUSTRAÇÃO</td></tr><tr...,627,2b2f97ca48af84c231330dc35a6fa1d1525371608d27bf84f26abae2928d0aca.jpg,"[70, 531, 527, 699]"
6,ggh9,GGH9.pdf,para_blocks,5,6,8,table,0.9877,,<table><tr><td rowspan=1 colspan=1>ESCOPO“b”</td><td rowspan=1 colspan=1>ESCOPO “C”APÓS AVALIAÇÃO DACONDIÇÃO E &quot;Pit Stop&quot;</td></tr><tr><td rowspan...,927,20545a34cec47b78d597a4ea5dc7323871094f252d694590eca2045513608c61.jpg,"[109, 554, 488, 657]"
7,ggh9,GGH9.pdf,para_blocks,6,7,8,table,0.9884,,<table><tr><td>PLANTA</td><td>ESCOPO</td><td>TEMPO DA PARADA SEM &quot;Pit Stop”</td><td>TEMPO DA PARADA COM “Pit Stop&quot;</td><td>ILUSTRAÇÃO</td></tr><tr...,627,2b2f97ca48af84c231330dc35a6fa1d1525371608d27bf84f26abae2928d0aca.jpg,"[70, 531, 527, 699]"
8,ggt_1247,GGT_1247.pdf,preproc_blocks,1,2,9,table,0.9892,,<table><tr><td>Nome do Projeto</td><td>Responsável</td><td>País</td><td>Ano</td><td>Foco</td></tr><tr><td>HIGGS</td><td>EU/FCH JU</td><td>Europa</td><td>202...,1794,637131c1fea824342bf4899fbe495a4032e2d5e334a35a2056b420de7c6df51e.jpg,"[76, 487, 521, 746]"
9,ggt_1247,GGT_1247.pdf,preproc_blocks,2,3,2,table,0.9592,,<table><tr><td>Ameland</td><td>GasTerra/Kiwa</td><td>Holanda</td><td>2007-2011</td><td>Demonstração e uso final</td></tr><tr><td>Naturalhy</td><td>Gaz de Fr...,248,51bb395b4dffc25e9b7035d41df1a037d6d82dcf1ccc01e3d10bab187280685d.jpg,"[76, 88, 521, 117]"


In [27]:
if not tables_df.empty:
    table_summary = tables_df.groupby(
        "filename",
        as_index=False,
    ).agg(
        table_count=(
            "block_index",
            "count",
        ),
        tables_with_html=(
            "html_length",
            lambda values: int((values > 0).sum()),
        ),
        tables_without_html=(
            "html_length",
            lambda values: int((values == 0).sum()),
        ),
    )
else:
    table_summary = pd.DataFrame(
        columns=[
            "filename",
            "table_count",
            "tables_with_html",
            "tables_without_html",
        ]
    )

table_summary

,filename,table_count,tables_with_html,tables_without_html
0,GDI_0685.pdf,4,4,0
1,GGH9.pdf,4,4,0
2,GGT_1247.pdf,10,9,1
3,GTM10.pdf,2,2,0


In [28]:
for row in tables_df.itertuples(index=False):
    if not row.html:
        continue

    display(Markdown(f"### {row.filename} — página {row.page_number}"))

    display(Markdown(row.html))

### GDI_0685.pdf — página 2

<table><tr><td rowspan=1 colspan=1>Distribuiçãode carga</td><td rowspan=1 colspan=1>VeículoCompleto(kg)</td><td rowspan=1 colspan=1>Peso BrutoTotal (kg)</td><td rowspan=1 colspan=1>CONTRAN(kg)</td></tr><tr><td rowspan=1 colspan=1>Eixo dianteiro</td><td rowspan=1 colspan=1>2.966</td><td rowspan=1 colspan=1>3.650</td><td rowspan=1 colspan=1>6.000</td></tr><tr><td rowspan=1 colspan=1>Eixo traseiro</td><td rowspan=1 colspan=1>4.474</td><td rowspan=1 colspan=1>8.150</td><td rowspan=1 colspan=1>10.000</td></tr><tr><td rowspan=1 colspan=1>Total</td><td rowspan=1 colspan=1>7.440</td><td rowspan=1 colspan=1>11.800</td><td rowspan=1 colspan=1>16.000</td></tr></table>

### GDI_0685.pdf — página 7

<table><tr><td></td><td>Limeira</td><td>Mairiporã</td></tr><tr><td>Número de dias em movimento</td><td>32</td><td>40</td></tr><tr><td>Distância total percorrida (km)</td><td>1.588</td><td>1.855</td></tr><tr><td>Distância média percorrida (km/dia)</td><td>49</td><td>46</td></tr><tr><td>Distância máxima percorrida (km/dia)</td><td>103</td><td>210</td></tr><tr><td>Tempo total ignição ligada (h)</td><td>368h 29min</td><td>405h</td></tr><tr><td>Tempo médio ignição ligada (h/dia)</td><td>9h 41min</td><td>8h 16min</td></tr><tr><td>Tempo máximo ignição ligada (h/dia)</td><td>23h 59min</td><td>17h 52min</td></tr><tr><td>Tempo total em movimento (h)</td><td>56h 16min</td><td>94h</td></tr><tr><td>Tempo médio em movimento (h/dia)</td><td>1h 47min</td><td>2h 20min</td></tr><tr><td>Tempo máximo em movimento (h/dia)</td><td>3h 32min</td><td>4h 35min</td></tr></table>

### GDI_0685.pdf — página 2

...

<table><tr><td rowspan=2 colspan=9>HCNG                                 H2 [% vol]         0%      20%      40%      60%       80%      100%CO2 evitado</td></tr><tr><td rowspan=1 colspan=1>[tcO2eq/ano]</td><td rowspan=1 colspan=3>0%7%17%</td><td rowspan=1 colspan=3>31%55%100%</td></tr><tr><td rowspan=1 colspan=2>CONSUMO FINAL ENERGÉTICO</td><td rowspan=1 colspan=1>19.460</td><td rowspan=1 colspan=1></td><td rowspan=1 colspan=1>2.476.937</td><td rowspan=1 colspan=1>5.913.926</td><td rowspan=1 colspan=1>11.003.315</td><td rowspan=1 colspan=1>19.313.836</td><td rowspan=1 colspan=1>35.319.280</td></tr><tr><td rowspan=3 colspan=2>SETOR ENERGÉTICORESIDENCIALCOMERCIAL</td><td rowspan=1 colspan=1>7.112</td><td rowspan=1 colspan=1>-</td><td rowspan=1 colspan=1>905.234</td><td rowspan=1 colspan=1>2.161.334</td><td rowspan=1 colspan=1>4.021.328</td><td rowspan=1 colspan=1>7.058.534</td><td rowspan=1 colspan=1>12.907.966</td></tr><tr><td rowspan=1 colspan=1>464</td><td rowspan=1 colspan=1></td><td rowspan=1 colspan=1>59.008</td><td rowspan=1 colspan=1>140.888</td><td rowspan=1 colspan=1>262.133</td><td rowspan=1 colspan=1>460.116</td><td rowspan=1 colspan=1>841.416</td></tr><tr><td rowspan=1 colspan=1>136</td><td rowspan=1 colspan=1></td><td rowspan=1 colspan=1>17.268</td><td rowspan=1 colspan=1>41.230</td><td rowspan=1 colspan=1>76.712</td><td rowspan=1 colspan=1>134.650</td><td rowspan=1 colspan=1>246.235</td></tr><tr><td rowspan=2 colspan=2>PÚBLICOTRANSPORTES - TOTAL</td><td rowspan=1 colspan=1>31</td><td rowspan=1 colspan=1></td><td rowspan=1 colspan=1>3.999</td><td rowspan=1 colspan=1>9.548</td><td rowspan=1 colspan=1>17.764</td><td rowspan=1 colspan=1>31.181</td><td rowspan=1 colspan=1>57.021</td></tr><tr><td rowspan=1 colspan=1></td><td rowspan=1 colspan=1>2.285</td><td rowspan=1 colspan=1></td><td rowspan=1 colspan=1>290.796</td><td rowspan=1 colspan=1>694.303</td><td rowspan=1 colspan=1>1.291.804</td><td rowspan=1 colspan=1>2.267.470</td><td rowspan=1 colspan=1>4.146.530</td></tr><tr><td rowspan=2 colspan=2>INDUSTRIAL - TOTALTOTAL IND + COM+ RES+GNV:</td><td rowspan=1 colspan=1>9.433</td><td rowspan=1 colspan=1></td><td rowspan=1 colspan=1>1.200.631</td><td rowspan=1 colspan=1>2.866.624</td><td rowspan=1 colspan=1>5.333.574</td><td rowspan=1 colspan=1>9.361.885</td><td rowspan=1 colspan=1>17.120.112</td></tr><tr><td rowspan=1 colspan=1>12.316</td><td rowspan=1 colspan=1></td><td rowspan=1 colspan=1>1.567.704</td><td rowspan=1 colspan=1>3.743.045</td><td rowspan=1 colspan=1>6.964.223</td><td rowspan=1 colspan=1>12.224.121</td><td rowspan=1 colspan=1>22.354.293</td></tr></table>

### GTM10.pdf — página 5

<table><tr><td rowspan=1 colspan=1>Espessuradablindagem(shunt)[mm]</td><td rowspan=1 colspan=1>Comprimentoda blindagem(shunt)[mm]</td><td rowspan=1 colspan=1>Perdasinduzidasno tanque[W]</td><td rowspan=1 colspan=1>Induçãomáximanablindagem[T]</td><td rowspan=1 colspan=1>Elevação detemperaturaestimada nocentro do tanque[C]</td><td rowspan=1 colspan=1>Elevação de temperaturaestimada do tanquepróximo à extremidade dablindagem[°c]</td></tr><tr><td rowspan=1 colspan=1>17</td><td rowspan=1 colspan=1>2800</td><td rowspan=1 colspan=1>23319</td><td rowspan=1 colspan=1>2,03</td><td rowspan=1 colspan=1>21</td><td rowspan=1 colspan=1>35,8</td></tr><tr><td rowspan=1 colspan=1>26</td><td rowspan=1 colspan=1>2800</td><td rowspan=1 colspan=1>20142</td><td rowspan=1 colspan=1>1,52</td><td rowspan=1 colspan=1>1,1</td><td rowspan=1 colspan=1>43,5</td></tr><tr><td rowspan=1 colspan=1>26</td><td rowspan=1 colspan=1>3300</td><td rowspan=1 colspan=1>5697</td><td rowspan=1 colspan=1>1,55</td><td rowspan=1 colspan=1>0,2</td><td rowspan=1 colspan=1>9,3</td></tr></table>

### GTM10.pdf — página 5

<table><tr><td rowspan=1 colspan=1>Espessuradablindagem(shunt)[mm]</td><td rowspan=1 colspan=1>Comprimentoda blindagem(shunt)[mm]</td><td rowspan=1 colspan=1>Perdasinduzidasno tanque[W]</td><td rowspan=1 colspan=1>Induçãomáximanablindagem[T]</td><td rowspan=1 colspan=1>Elevação detemperaturaestimada nocentro do tanque[C]</td><td rowspan=1 colspan=1>Elevação de temperaturaestimada do tanquepróximo à extremidade dablindagem[°c]</td></tr><tr><td rowspan=1 colspan=1>17</td><td rowspan=1 colspan=1>2800</td><td rowspan=1 colspan=1>23319</td><td rowspan=1 colspan=1>2,03</td><td rowspan=1 colspan=1>21</td><td rowspan=1 colspan=1>35,8</td></tr><tr><td rowspan=1 colspan=1>26</td><td rowspan=1 colspan=1>2800</td><td rowspan=1 colspan=1>20142</td><td rowspan=1 colspan=1>1,52</td><td rowspan=1 colspan=1>1,1</td><td rowspan=1 colspan=1>43,5</td></tr><tr><td rowspan=1 colspan=1>26</td><td rowspan=1 colspan=1>3300</td><td rowspan=1 colspan=1>5697</td><td rowspan=1 colspan=1>1,55</td><td rowspan=1 colspan=1>0,2</td><td rowspan=1 colspan=1>9,3</td></tr></table>

## 15. Fórmulas

In [29]:
FORMULA_TYPES = {
    "equation",
    "display_formula",
    "inline_formula",
    "interline_equation",
    "equation_interline",
    "equation_inline",
    "formula",
    "formula_number",
}


formula_rows: list[dict[str, Any]] = []

for document_id, document_record in loaded_documents.items():
    document = document_record["data"]

    for block_collection in BLOCK_FIELDS:
        for (
            page_index,
            block_index,
            block,
        ) in iter_blocks(
            document,
            block_collection,
        ):
            nested_latex = find_nested_values(
                block,
                {
                    "latex",
                    "math_content",
                },
            )

            formula_spans = []

            for _, line in iter_lines(block):
                for _, span in iter_spans(line):
                    if span.get("type") in FORMULA_TYPES or span.get("latex"):
                        formula_spans.append(span)

            is_formula = (
                block.get("type") in FORMULA_TYPES
                or bool(nested_latex)
                or bool(formula_spans)
            )

            if not is_formula:
                continue

            latex_values = [
                str(value)
                for _, value in nested_latex
                if isinstance(
                    value,
                    str,
                )
                and value.strip()
            ]

            latex_values.extend(
                str(span.get("latex") or span.get("content") or "")
                for span in formula_spans
                if (span.get("latex") or span.get("content"))
            )

            formula_rows.append(
                {
                    "document_id": document_id,
                    "filename": document_record["filename"],
                    "block_collection": (block_collection),
                    "page_index": page_index,
                    "page_number": (page_index + 1),
                    "block_index": block_index,
                    "block_type": block.get("type"),
                    "score": block.get("score"),
                    "latex": "\n\n".join(dict.fromkeys(latex_values)),
                    "latex_length": sum(len(value) for value in latex_values),
                    "text": (extract_text_from_block(block)),
                    "bbox": block.get("bbox"),
                }
            )


formulas_df = pd.DataFrame(formula_rows)

formulas_df

,document_id,filename,block_collection,page_index,page_number,block_index,block_type,score,latex,latex_length,text,bbox
0,ggt_1247,GGT_1247.pdf,preproc_blocks,3,4,6,interline_equation,0.9727,\begin{array} { r l } & { ( 1 - f _ { H _ { 2 } } ) \cdot C H _ { 4 } + f _ { H _ { 2 } } \cdot H _ { 2 } + \lambda \cdot [ 2 \cdot ( 1 - f _ { H _ { 2 } } ...,545,\begin{array} { r l } & { ( 1 - f _ { H _ { 2 } } ) \cdot C H _ { 4 } + f _ { H _ { 2 } } \cdot H _ { 2 } + \lambda \cdot [ 2 \cdot ( 1 - f _ { H _ { 2 } } ...,"[90, 259, 492, 304]"
1,ggt_1247,GGT_1247.pdf,preproc_blocks,3,4,10,interline_equation,0.9589,"C O _ { 2 } [ \% v o l ] = \frac { ( 1 - f _ { H _ { 2 } } ) } { 9 , 5 2 \lambda + 0 , 5 f _ { H _ { 2 } } - 7 , 1 4 \lambda f _ { H _ { 2 } } - 1 }\tag{2}",155,"C O _ { 2 } [ \% v o l ] = \frac { ( 1 - f _ { H _ { 2 } } ) } { 9 , 5 2 \lambda + 0 , 5 f _ { H _ { 2 } } - 7 , 1 4 \lambda f _ { H _ { 2 } } - 1 }\tag{2}","[220, 388, 395, 413]"
2,ggt_1247,GGT_1247.pdf,preproc_blocks,3,4,13,interline_equation,0.9600,\frac { V _ { C O _ { 2 } } } { E _ { f } } = \frac { ( 1 - f _ { H _ { 2 } } ) } { f _ { H _ { 2 } } L H V _ { H _ { 2 } } + \left( 1 - f _ { H _ { 2 } } \...,195,\frac { V _ { C O _ { 2 } } } { E _ { f } } = \frac { ( 1 - f _ { H _ { 2 } } ) } { f _ { H _ { 2 } } L H V _ { H _ { 2 } } + \left( 1 - f _ { H _ { 2 } } \...,"[235, 464, 378, 491]"
3,ggt_1247,GGT_1247.pdf,preproc_blocks,3,4,15,interline_equation,0.9572,\frac { m _ { C O _ { 2 } } } { E _ { f } } = \frac { \rho _ { C O _ { 2 } } ( 1 - f _ { H _ { 2 } } ) } { f _ { H _ { 2 } } L H V _ { H _ { 2 } } + \big ( ...,217,\frac { m _ { C O _ { 2 } } } { E _ { f } } = \frac { \rho _ { C O _ { 2 } } ( 1 - f _ { H _ { 2 } } ) } { f _ { H _ { 2 } } L H V _ { H _ { 2 } } + \big ( ...,"[234, 499, 379, 525]"
4,ggt_1247,GGT_1247.pdf,para_blocks,3,4,6,interline_equation,0.9727,\begin{array} { r l } & { ( 1 - f _ { H _ { 2 } } ) \cdot C H _ { 4 } + f _ { H _ { 2 } } \cdot H _ { 2 } + \lambda \cdot [ 2 \cdot ( 1 - f _ { H _ { 2 } } ...,545,\begin{array} { r l } & { ( 1 - f _ { H _ { 2 } } ) \cdot C H _ { 4 } + f _ { H _ { 2 } } \cdot H _ { 2 } + \lambda \cdot [ 2 \cdot ( 1 - f _ { H _ { 2 } } ...,"[90, 259, 492, 304]"
5,ggt_1247,GGT_1247.pdf,para_blocks,3,4,10,interline_equation,0.9589,"C O _ { 2 } [ \% v o l ] = \frac { ( 1 - f _ { H _ { 2 } } ) } { 9 , 5 2 \lambda + 0 , 5 f _ { H _ { 2 } } - 7 , 1 4 \lambda f _ { H _ { 2 } } - 1 }\tag{2}",155,"C O _ { 2 } [ \% v o l ] = \frac { ( 1 - f _ { H _ { 2 } } ) } { 9 , 5 2 \lambda + 0 , 5 f _ { H _ { 2 } } - 7 , 1 4 \lambda f _ { H _ { 2 } } - 1 }\tag{2}","[220, 388, 395, 413]"
6,ggt_1247,GGT_1247.pdf,para_blocks,3,4,13,interline_equation,0.9600,\frac { V _ { C O _ { 2 } } } { E _ { f } } = \frac { ( 1 - f _ { H _ { 2 } } ) } { f _ { H _ { 2 } } L H V _ { H _ { 2 } } + \left( 1 - f _ { H _ { 2 } } \...,195,\frac { V _ { C O _ { 2 } } } { E _ { f } } = \frac { ( 1 - f _ { H _ { 2 } } ) } { f _ { H _ { 2 } } L H V _ { H _ { 2 } } + \left( 1 - f _ { H _ { 2 } } \...,"[235, 464, 378, 491]"
7,ggt_1247,GGT_1247.pdf,para_blocks,3,4,15,interline_equation,0.9572,\frac { m _ { C O _ { 2 } } } { E _ { f } } = \frac { \rho _ { C O _ { 2 } } ( 1 - f _ { H _ { 2 } } ) } { f _ { H _ { 2 } } L H V _ { H _ { 2 } } + \big ( ...,217,\frac { m _ { C O _ { 2 } } } { E _ { f } } = \frac { \rho _ { C O _ { 2 } } ( 1 - f _ { H _ { 2 } } ) } { f _ { H _ { 2 } } L H V _ { H _ { 2 } } + \big ( ...,"[234, 499, 379, 525]"


In [30]:
if not formulas_df.empty:
    formula_summary = formulas_df.groupby(
        "filename",
        as_index=False,
    ).agg(
        formula_count=(
            "block_index",
            "count",
        ),
        formulas_with_latex=(
            "latex_length",
            lambda values: int((values > 0).sum()),
        ),
        formulas_without_latex=(
            "latex_length",
            lambda values: int((values == 0).sum()),
        ),
    )
else:
    formula_summary = pd.DataFrame(
        columns=[
            "filename",
            "formula_count",
            "formulas_with_latex",
            "formulas_without_latex",
        ]
    )

formula_summary

,filename,formula_count,formulas_with_latex,formulas_without_latex
0,GGT_1247.pdf,8,8,0


In [31]:
for row in formulas_df.itertuples(index=False):
    if not row.latex:
        continue

    display(Markdown(f"### {row.filename} — página {row.page_number}"))

    display(Markdown(f"```latex\n{row.latex}\n```"))

### GGT_1247.pdf — página 4

```latex
\begin{array} { r l } & { ( 1 - f _ { H _ { 2 } } ) \cdot C H _ { 4 } + f _ { H _ { 2 } } \cdot H _ { 2 } + \lambda \cdot [ 2 \cdot ( 1 - f _ { H _ { 2 } } ) + \displaystyle \frac { f _ { H _ { 2 } } } { 2 } ] \cdot ( { \cal O } _ { 2 } + 3 . 7 6 N _ { 2 } ) } \\ & { \qquad ( 1 - f _ { H _ { 2 } } ) \cdot C O _ { 2 } + ( 2 - f _ { H _ { 2 } } ) \cdot H _ { 2 } O + [ 2 \cdot ( 1 - f _ { H _ { 2 } } ) + \displaystyle \frac { f _ { H _ { 2 } } } { 2 } ] \cdot [ ( \lambda - 1 ) \cdot O _ { 2 } + \lambda 3 . 7 6 N _ { 2 } ] } \end{array}\tag{1}
```

### GGT_1247.pdf — página 4

```latex
C O _ { 2 } [ \% v o l ] = \frac { ( 1 - f _ { H _ { 2 } } ) } { 9 , 5 2 \lambda + 0 , 5 f _ { H _ { 2 } } - 7 , 1 4 \lambda f _ { H _ { 2 } } - 1 }\tag{2}
```

### GGT_1247.pdf — página 4

...

```latex
C O _ { 2 } [ \% v o l ] = \frac { ( 1 - f _ { H _ { 2 } } ) } { 9 , 5 2 \lambda + 0 , 5 f _ { H _ { 2 } } - 7 , 1 4 \lambda f _ { H _ { 2 } } - 1 }\tag{2}
```

### GGT_1247.pdf — página 4

```latex
\frac { V _ { C O _ { 2 } } } { E _ { f } } = \frac { ( 1 - f _ { H _ { 2 } } ) } { f _ { H _ { 2 } } L H V _ { H _ { 2 } } + \left( 1 - f _ { H _ { 2 } } \right) L H V _ { C H _ { 4 } } }\tag{3}
```

### GGT_1247.pdf — página 4

```latex
\frac { m _ { C O _ { 2 } } } { E _ { f } } = \frac { \rho _ { C O _ { 2 } } ( 1 - f _ { H _ { 2 } } ) } { f _ { H _ { 2 } } L H V _ { H _ { 2 } } + \big ( 1 - f _ { H _ { 2 } } \big ) L H V _ { C H _ { 4 } } }\tag{4}
```

## 16. Figuras, gráficos e imagens

In [32]:
IMAGE_TYPES = {
    "figure",
    "image",
    "chart",
    "figure_body",
    "header_image",
}


image_rows: list[dict[str, Any]] = []

for document_id, document_record in loaded_documents.items():
    document = document_record["data"]

    for block_collection in BLOCK_FIELDS:
        for (
            page_index,
            block_index,
            block,
        ) in iter_blocks(
            document,
            block_collection,
        ):
            image_values = [
                value
                for _, value in find_nested_values(
                    block,
                    {
                        "image_path",
                        "img_path",
                    },
                )
                if value
            ]

            is_image = block.get("type") in IMAGE_TYPES or bool(image_values)

            if not is_image:
                continue

            image_rows.append(
                {
                    "document_id": document_id,
                    "filename": document_record["filename"],
                    "block_collection": (block_collection),
                    "page_index": page_index,
                    "page_number": (page_index + 1),
                    "block_index": block_index,
                    "block_type": block.get("type"),
                    "score": block.get("score"),
                    "image_paths": (
                        " | ".join(
                            map(
                                str,
                                image_values,
                            )
                        )
                    ),
                    "text": (extract_text_from_block(block)),
                    "bbox": block.get("bbox"),
                }
            )


images_df = pd.DataFrame(image_rows)

images_df

,document_id,filename,block_collection,page_index,page_number,block_index,block_type,score,image_paths,text,bbox
0,gdi_0685,GDI_0685.pdf,preproc_blocks,1,2,2,image,0.9871,196ae709b1349e325272ed6018ae96464560f8fa7303afd0edb42d2a2a754d9e.jpg,,"[191, 84, 407, 223]"
1,gdi_0685,GDI_0685.pdf,preproc_blocks,1,2,6,image,0.9798,1228544f4c78c48c9308698d79092b6b9540e321b548a23a6f85a987183ccfe7.jpg,,"[142, 463, 453, 643]"
2,gdi_0685,GDI_0685.pdf,preproc_blocks,1,2,9,table,0.9851,31c09c6d92f6710149776c78dce50432ad53ea940064bf1b4d4864a9f5c4bbbc.jpg,,"[194, 681, 403, 754]"
3,gdi_0685,GDI_0685.pdf,preproc_blocks,2,3,9,image,0.9653,b5e6440d55206619f6812d3bb5f0d70b9ce59712fb2c32f2fa2cdd5e0d873c3d.jpg,,"[119, 446, 478, 611]"
4,gdi_0685,GDI_0685.pdf,preproc_blocks,3,4,3,image,0.9850,e0ec3dde4e586867fe638f31c222020a29f665e042608023d1317b5ba81d258b.jpg,,"[99, 131, 498, 266]"
...,...,...,...,...,...,...,...,...,...,...,...
138,gtm10,GTM10.pdf,para_blocks,6,7,2,chart,0.9842,9d98b5709ac10f3d2b79e0df1b1483117052976a76f6f1bccf00c2f46245c5ff.jpg,,"[137, 85, 477, 255]"
139,gtm10,GTM10.pdf,para_blocks,8,9,3,image,0.9746,18dd45cd9029d64090e366c331adfacc782f882f4ff3b0d9b8f94d315fcd5cce.jpg,,"[66, 107, 135, 190]"
140,gtm10,GTM10.pdf,para_blocks,8,9,5,image,0.9647,8b9beb218814ab2af1d30ae6f1124452d051007d05b5e292ebf9a15b28b29a39.jpg,,"[66, 216, 138, 295]"
141,gtm10,GTM10.pdf,para_blocks,8,9,8,image,0.9627,0853f433e7763047a92b3351261b8149828c947ca55838637651df8388dbe80d.jpg,,"[70, 352, 136, 431]"


## 17. Blocos descartados

Cabeçalhos, rodapés e números de página devem aparecer principalmente em
`discarded_blocks`.

In [33]:
discarded_df = (
    blocks_df[blocks_df["block_collection"] == "discarded_blocks"][
        [
            "filename",
            "page_number",
            "block_index",
            "block_type",
            "score",
            "line_count",
            "span_count",
            "text",
            "x0",
            "y0",
            "x1",
            "y1",
        ]
    ]
    .sort_values(
        [
            "filename",
            "page_number",
            "block_index",
        ]
    )
    .reset_index(drop=True)
)

discarded_df

,filename,page_number,block_index,block_type,score,line_count,span_count,text,x0,y0,x1,y1
0,GDI_0685.pdf,1,1,header,0.9276,4,4,XXVII Seminário Nacional de Produção e Transmissão de Energia Elétrica XXVIISNPTEE,66.0,34.0,250.0,76.0
1,GDI_0685.pdf,1,2,header,0.8573,2,2,0685 GDI/7,497.0,39.0,525.0,67.0
2,GDI_0685.pdf,1,17,footer,0.9304,1,1,ana.oening@lactec.com.br,240.0,789.0,358.0,801.0
3,GDI_0685.pdf,2,1,page_number,0.8665,1,1,2,295.0,36.0,303.0,45.0
4,GDI_0685.pdf,3,1,page_number,0.8621,1,1,3,295.0,36.0,303.0,45.0
...,...,...,...,...,...,...,...,...,...,...,...,...
56,GTM10.pdf,5,1,page_number,0.8699,1,1,5,304.0,41.0,311.0,51.0
57,GTM10.pdf,6,1,page_number,0.8728,1,1,6,304.0,41.0,311.0,51.0
58,GTM10.pdf,7,1,page_number,0.8486,1,1,7,304.0,41.0,311.0,51.0
59,GTM10.pdf,8,1,page_number,0.8620,1,1,8,304.0,41.0,311.0,51.0


In [34]:
discarded_type_counts = (
    discarded_df.groupby(
        "block_type",
        dropna=False,
    )
    .size()
    .rename("count")
    .reset_index()
    .sort_values(
        "count",
        ascending=False,
    )
    .reset_index(drop=True)
)

discarded_type_counts

,block_type,count
0,page_number,36
1,header,14
2,footer,11


## 18. Cabeçalhos, rodapés e números de página fora de `discarded_blocks`

Esta verificação identifica elementos potencialmente vazando para o conteúdo
principal.

In [35]:
NOISE_TYPES = {
    "header",
    "footer",
    "page_number",
    "number",
}


noise_blocks = (
    blocks_df[blocks_df["block_type"].isin(NOISE_TYPES)][
        [
            "filename",
            "block_collection",
            "page_number",
            "block_index",
            "block_type",
            "score",
            "text",
        ]
    ]
    .sort_values(
        [
            "filename",
            "page_number",
            "block_index",
            "block_collection",
        ]
    )
    .reset_index(drop=True)
)

noise_blocks

,filename,block_collection,page_number,block_index,block_type,score,text
0,GDI_0685.pdf,discarded_blocks,1,1,header,0.9276,XXVII Seminário Nacional de Produção e Transmissão de Energia Elétrica XXVIISNPTEE
1,GDI_0685.pdf,discarded_blocks,1,2,header,0.8573,0685 GDI/7
2,GDI_0685.pdf,discarded_blocks,1,17,footer,0.9304,ana.oening@lactec.com.br
3,GDI_0685.pdf,discarded_blocks,2,1,page_number,0.8665,2
4,GDI_0685.pdf,discarded_blocks,3,1,page_number,0.8621,3
...,...,...,...,...,...,...,...
56,GTM10.pdf,discarded_blocks,5,1,page_number,0.8699,5
57,GTM10.pdf,discarded_blocks,6,1,page_number,0.8728,6
58,GTM10.pdf,discarded_blocks,7,1,page_number,0.8486,7
59,GTM10.pdf,discarded_blocks,8,1,page_number,0.8620,8


In [36]:
noise_leaks = noise_blocks[
    noise_blocks["block_collection"] != "discarded_blocks"
].copy()

noise_leaks

,filename,block_collection,page_number,block_index,block_type,score,text


## 19. Continuidade entre páginas

In [37]:
cross_page_spans = (
    spans_df[spans_df["cross_page"]][
        [
            "filename",
            "block_collection",
            "page_number",
            "block_index",
            "block_type",
            "line_index",
            "span_index",
            "span_type",
            "span_text",
            "score",
        ]
    ]
    .sort_values(
        [
            "filename",
            "page_number",
            "block_index",
            "line_index",
            "span_index",
        ]
    )
    .reset_index(drop=True)
)

cross_page_spans

,filename,block_collection,page_number,block_index,block_type,line_index,span_index,span_type,span_text,score
0,GGH9.pdf,para_blocks,2,9,text,5,0,text,"(quinze) dias de perda de receitas equivalerem ao preço de uma nova roda de turbina, valerá mais a pena investir",1.0
1,GGH9.pdf,para_blocks,2,9,text,6,0,text,na compra da nova roda e realizar uma parada rápida somente para troca da roda do que realizar o serviço de,1.0
2,GGH9.pdf,para_blocks,2,9,text,7,0,text,menor preço de recuperação da turbina existente em campo. A situação pode ser bastante onerosa também no,1.0
3,GGH9.pdf,para_blocks,2,9,text,8,0,text,"caso de turbinas de maior porte e/ou em localidades mais remotas, principalmente em unidades mais antigas, pois",1.0
4,GGH9.pdf,para_blocks,2,9,text,9,0,text,os processos de desmontagem/montagem e transporte de todas as partes para uma fábrica podem levar a meses,1.0
5,GGH9.pdf,para_blocks,2,9,text,10,0,text,de indisponibilidade de uma grande capacidade instalada de geração. Tudo se torna uma questão de análise,1.0
6,GGH9.pdf,para_blocks,2,9,text,11,0,text,"completa do cenário de qualidade, custo e prazo. Desta forma, é importante buscar estratégias que minimizem as",1.0
7,GGH9.pdf,para_blocks,2,9,text,12,0,text,"paradas, mesmo que o investimento no serviço em si possa ficar maior.",1.0
8,GGH9.pdf,para_blocks,5,12,text,11,0,text,"isto também resultou em ganhos nos tempos de montagem e comissionamento. Logo, obteve-se uma redução de",1.0
9,GGH9.pdf,para_blocks,5,12,text,12,0,text,"18 (dezoito) dias no processo de montagem e comissionamento, o que representou em uma redução de 45% nos",1.0


In [38]:
cross_page_summary = (
    cross_page_spans.groupby(
        "filename",
        as_index=False,
    ).agg(
        cross_page_span_count=(
            "span_index",
            "count",
        ),
        affected_pages=(
            "page_number",
            "nunique",
        ),
    )
    if not cross_page_spans.empty
    else pd.DataFrame(
        columns=[
            "filename",
            "cross_page_span_count",
            "affected_pages",
        ]
    )
)

cross_page_summary

,filename,cross_page_span_count,affected_pages
0,GGH9.pdf,12,2
1,GGT_1247.pdf,2,1
2,GTM10.pdf,2,1


## 20. Scores dos blocos e spans

In [39]:
block_score_summary = (
    blocks_df.groupby(
        [
            "block_collection",
            "block_type",
        ],
        dropna=False,
    )
    .agg(
        count=(
            "score",
            "size",
        ),
        score_count=(
            "score",
            "count",
        ),
        score_min=(
            "score",
            "min",
        ),
        score_mean=(
            "score",
            "mean",
        ),
        score_median=(
            "score",
            "median",
        ),
        score_max=(
            "score",
            "max",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "block_collection",
            "score_mean",
        ],
        ascending=[
            True,
            True,
        ],
    )
)

block_score_summary

,block_collection,block_type,count,score_count,score_min,score_mean,score_median,score_max
2,discarded_blocks,page_number,36,36,0.8167,0.861150,0.86205,0.8919
1,discarded_blocks,header,14,14,0.4931,0.867157,0.90635,0.9728
0,discarded_blocks,footer,11,11,0.8980,0.931527,0.93570,0.9565
7,para_blocks,list,1,1,0.7053,0.705300,0.70530,0.7053
11,para_blocks,title,69,69,0.5168,0.887549,0.93510,0.9671
8,para_blocks,ref_text,48,48,0.7939,0.949717,0.95970,0.9768
10,para_blocks,text,215,215,0.4600,0.956455,0.97990,0.9887
6,para_blocks,interline_equation,4,4,0.9572,0.962200,0.95945,0.9727
5,para_blocks,image,38,38,0.9047,0.970453,0.97755,0.9886
3,para_blocks,abstract,3,3,0.9681,0.972033,0.96820,0.9798


In [40]:
LOW_SCORE_THRESHOLD = 0.80


low_score_blocks = (
    blocks_df[blocks_df["score"].notna() & (blocks_df["score"] < LOW_SCORE_THRESHOLD)][
        [
            "filename",
            "block_collection",
            "page_number",
            "block_index",
            "block_type",
            "score",
            "text",
        ]
    ]
    .sort_values(
        [
            "score",
            "filename",
            "page_number",
        ]
    )
    .reset_index(drop=True)
)

low_score_blocks

,filename,block_collection,page_number,block_index,block_type,score,text
0,GTM10.pdf,preproc_blocks,1,9,text,0.4600,Luiz Fernando de Oliveira (*) Álvaro Portillo
1,GTM10.pdf,para_blocks,1,9,text,0.4600,Luiz Fernando de Oliveira (*) Álvaro Portillo
2,GTL_0081.pdf,preproc_blocks,5,7,text,0.4844,(3) SIDNEY FERREIRA PINTO
3,GTL_0081.pdf,para_blocks,5,7,text,0.4844,(3) SIDNEY FERREIRA PINTO
4,GGT_1247.pdf,preproc_blocks,5,2,text,0.4853,Figura 1 - Comparação do comportamento dos poderes caloríficos por unidade de massa e volume de acordo com a porcentagem de hidrogênio na mistura.
5,GGT_1247.pdf,para_blocks,5,2,text,0.4853,Figura 1 - Comparação do comportamento dos poderes caloríficos por unidade de massa e volume de acordo com a porcentagem de hidrogênio na mistura.
6,GTM10.pdf,discarded_blocks,1,4,header,0.4931,22 a 25 de outubro de 2017
7,GTM10.pdf,preproc_blocks,2,4,title,0.5168,2.0 - COMPARATIVO ENTRE OS MÉTODOS EXPONENCIAL E DIFERENCIAL
8,GTM10.pdf,para_blocks,2,4,title,0.5168,2.0 - COMPARATIVO ENTRE OS MÉTODOS EXPONENCIAL E DIFERENCIAL
9,GTL_0081.pdf,preproc_blocks,1,4,title,0.5727,CENTRO VIRTUAL DE OPERAÇÃO


In [41]:
if spans_df.empty:
    span_score_summary = pd.DataFrame()
else:
    span_score_summary = (
        spans_df.groupby(
            "span_type",
            dropna=False,
        )
        .agg(
            count=(
                "score",
                "size",
            ),
            score_count=(
                "score",
                "count",
            ),
            score_min=(
                "score",
                "min",
            ),
            score_mean=(
                "score",
                "mean",
            ),
            score_median=(
                "score",
                "median",
            ),
            score_max=(
                "score",
                "max",
            ),
        )
        .reset_index()
        .sort_values(
            "score_mean",
            ascending=True,
        )
    )

span_score_summary

,span_type,count,score_count,score_min,score_mean,score_median,score_max
0,inline_equation,34,34,0.4563,0.680112,0.6651,0.8782
2,text,2585,2585,0.0000,0.999270,1.0000,1.0000
1,interline_equation,8,0,NaN,NaN,NaN,NaN


## 21. Campos encontrados nos blocos

Este inventário evita assumir prematuramente um schema fixo.

In [42]:
def collect_dict_keys(
    items: Iterable[dict[str, Any]],
) -> Counter[str]:
    counter: Counter[str] = Counter()

    for item in items:
        counter.update(item.keys())

    return counter


block_key_rows: list[dict[str, Any]] = []

for document_id, document_record in loaded_documents.items():
    document = document_record["data"]

    for block_collection in BLOCK_FIELDS:
        blocks = [
            block
            for _, _, block in iter_blocks(
                document,
                block_collection,
            )
        ]

        key_counts = collect_dict_keys(blocks)

        for key, count in key_counts.items():
            block_key_rows.append(
                {
                    "document_id": document_id,
                    "filename": document_record["filename"],
                    "block_collection": (block_collection),
                    "key": key,
                    "count": count,
                }
            )


block_keys_df = pd.DataFrame(block_key_rows)

block_keys_df

,document_id,filename,block_collection,key,count
0,gdi_0685,GDI_0685.pdf,preproc_blocks,score,88
1,gdi_0685,GDI_0685.pdf,preproc_blocks,bbox,88
2,gdi_0685,GDI_0685.pdf,preproc_blocks,index,88
3,gdi_0685,GDI_0685.pdf,preproc_blocks,type,88
4,gdi_0685,GDI_0685.pdf,preproc_blocks,lines,69
...,...,...,...,...,...
99,gtm10,GTM10.pdf,discarded_blocks,score,13
100,gtm10,GTM10.pdf,discarded_blocks,bbox,13
101,gtm10,GTM10.pdf,discarded_blocks,index,13
102,gtm10,GTM10.pdf,discarded_blocks,type,13


In [43]:
block_key_summary = (
    block_keys_df.groupby(
        [
            "block_collection",
            "key",
        ],
        as_index=False,
    )
    .agg(
        document_count=(
            "document_id",
            "nunique",
        ),
        occurrence_count=(
            "count",
            "sum",
        ),
    )
    .sort_values(
        [
            "block_collection",
            "occurrence_count",
            "key",
        ],
        ascending=[
            True,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

block_key_summary

,block_collection,key,document_count,occurrence_count
0,discarded_blocks,bbox,5,61
1,discarded_blocks,index,5,61
2,discarded_blocks,lines,5,61
3,discarded_blocks,score,5,61
4,discarded_blocks,type,5,61
5,para_blocks,bbox,5,408
6,para_blocks,index,5,408
7,para_blocks,score,5,408
8,para_blocks,type,5,408
9,para_blocks,lines,5,340


## 22. Campos encontrados em linhas e spans

In [44]:
line_key_counter: Counter[str] = Counter()
span_key_counter: Counter[str] = Counter()

for document_record in loaded_documents.values():
    document = document_record["data"]

    for block_collection in BLOCK_FIELDS:
        for _, _, block in iter_blocks(
            document,
            block_collection,
        ):
            for _, line in iter_lines(block):
                line_key_counter.update(line.keys())

                for _, span in iter_spans(line):
                    span_key_counter.update(span.keys())


line_keys_df = pd.DataFrame(
    [
        {
            "key": key,
            "count": count,
        }
        for key, count in line_key_counter.most_common()
    ]
)

span_keys_df = pd.DataFrame(
    [
        {
            "key": key,
            "count": count,
        }
        for key, count in span_key_counter.most_common()
    ]
)


line_keys_df

,key,count
0,bbox,2475
1,spans,2475
2,is_list_start_line,2
3,is_list_end_line,2


In [45]:
span_keys_df

,key,count
0,bbox,2627
1,type,2627
2,content,2627
3,score,2619
4,cross_page,16
5,image_path,8


## 23. Duplicação entre `preproc_blocks` e `para_blocks`

O `middle.json` costuma manter versões diferentes dos blocos antes e depois
do agrupamento em parágrafos. Esta comparação mede quanto conteúdo textual
aparece em ambas as coleções.

In [46]:
def normalized_text(
    value: Any,
) -> str:
    return " ".join(str(value or "").casefold().split())


duplicate_rows: list[dict[str, Any]] = []

for (
    filename,
    page_number,
), page_group in blocks_df.groupby(
    [
        "filename",
        "page_number",
    ]
):
    preproc = page_group[page_group["block_collection"] == "preproc_blocks"].copy()

    para = page_group[page_group["block_collection"] == "para_blocks"].copy()

    preproc["normalized_text"] = preproc["text"].map(normalized_text)

    para["normalized_text"] = para["text"].map(normalized_text)

    preproc = preproc[preproc["normalized_text"] != ""]

    para = para[para["normalized_text"] != ""]

    para_texts = set(para["normalized_text"])

    matching_preproc = preproc[preproc["normalized_text"].isin(para_texts)]

    duplicate_rows.append(
        {
            "filename": filename,
            "page_number": page_number,
            "preproc_text_blocks": len(preproc),
            "para_text_blocks": len(para),
            "exact_text_matches": len(matching_preproc),
            "preproc_match_ratio": (
                len(matching_preproc) / len(preproc) if len(preproc) else None
            ),
        }
    )


duplicate_summary = pd.DataFrame(duplicate_rows)

duplicate_summary

,filename,page_number,preproc_text_blocks,para_text_blocks,exact_text_matches,preproc_match_ratio
0,GDI_0685.pdf,1,14,14,14,1.000000
1,GDI_0685.pdf,2,2,2,2,1.000000
2,GDI_0685.pdf,3,9,9,9,1.000000
3,GDI_0685.pdf,4,12,12,12,1.000000
4,GDI_0685.pdf,5,2,2,2,1.000000
5,GDI_0685.pdf,6,5,5,5,1.000000
6,GDI_0685.pdf,7,11,11,11,1.000000
7,GDI_0685.pdf,8,5,5,5,1.000000
8,GDI_0685.pdf,9,9,9,9,1.000000
9,GGH9.pdf,1,13,13,13,1.000000


## 24. Possíveis anomalias estruturais

In [47]:
anomaly_rows: list[dict[str, Any]] = []


for row in blocks_df.itertuples(index=False):
    if not row.block_type:
        anomaly_rows.append(
            {
                "filename": row.filename,
                "page_number": row.page_number,
                "block_collection": (row.block_collection),
                "block_index": row.block_index,
                "anomaly": ("block_type_missing"),
                "details": truncate_text(row.text),
            }
        )

    if (
        row.text_length == 0
        and not row.has_html
        and not row.has_latex
        and not row.has_image_path
        and not row.has_blocks
    ):
        anomaly_rows.append(
            {
                "filename": row.filename,
                "page_number": row.page_number,
                "block_collection": (row.block_collection),
                "block_index": row.block_index,
                "anomaly": ("empty_block"),
                "details": (f"type={row.block_type}"),
            }
        )

    if row.has_bbox and (
        row.width is None or row.height is None or row.width <= 0 or row.height <= 0
    ):
        anomaly_rows.append(
            {
                "filename": row.filename,
                "page_number": row.page_number,
                "block_collection": (row.block_collection),
                "block_index": row.block_index,
                "anomaly": ("invalid_bbox"),
                "details": (f"bbox=[{row.x0}, {row.y0}, {row.x1}, {row.y1}]"),
            }
        )

    if row.block_type in NOISE_TYPES and row.block_collection != "discarded_blocks":
        anomaly_rows.append(
            {
                "filename": row.filename,
                "page_number": row.page_number,
                "block_collection": (row.block_collection),
                "block_index": row.block_index,
                "anomaly": ("noise_outside_discarded"),
                "details": truncate_text(row.text),
            }
        )


for row in tables_df.itertuples(index=False):
    if row.html_length == 0:
        anomaly_rows.append(
            {
                "filename": row.filename,
                "page_number": row.page_number,
                "block_collection": (row.block_collection),
                "block_index": row.block_index,
                "anomaly": ("table_without_html"),
                "details": truncate_text(row.caption),
            }
        )


for row in formulas_df.itertuples(index=False):
    if row.latex_length == 0:
        anomaly_rows.append(
            {
                "filename": row.filename,
                "page_number": row.page_number,
                "block_collection": (row.block_collection),
                "block_index": row.block_index,
                "anomaly": ("formula_without_latex"),
                "details": truncate_text(row.text),
            }
        )


anomalies_df = pd.DataFrame(anomaly_rows)

if not anomalies_df.empty:
    anomalies_df = anomalies_df.sort_values(
        [
            "filename",
            "page_number",
            "block_index",
            "anomaly",
        ]
    ).reset_index(drop=True)

anomalies_df

,filename,page_number,block_collection,block_index,anomaly,details
0,GGH9.pdf,3,para_blocks,2,empty_block,type=text
1,GGH9.pdf,6,para_blocks,2,empty_block,type=text
2,GGT_1247.pdf,2,para_blocks,2,empty_block,type=text
3,GGT_1247.pdf,3,para_blocks,2,table_without_html,
4,GTL_0081.pdf,5,para_blocks,6,empty_block,type=text
5,GTM10.pdf,8,para_blocks,2,empty_block,type=text


In [48]:
if anomalies_df.empty:
    anomaly_summary = pd.DataFrame(
        columns=[
            "anomaly",
            "count",
        ]
    )
else:
    anomaly_summary = (
        anomalies_df.groupby("anomaly")
        .size()
        .rename("count")
        .reset_index()
        .sort_values(
            "count",
            ascending=False,
        )
        .reset_index(drop=True)
    )

anomaly_summary

,anomaly,count
0,empty_block,5
1,table_without_html,1


## 25. Resumo consolidado por documento

In [49]:
document_summary = (
    page_summary.merge(
        table_summary,
        on="filename",
        how="left",
    )
    .merge(
        formula_summary,
        on="filename",
        how="left",
    )
    .merge(
        cross_page_summary,
        on="filename",
        how="left",
    )
)


for column in (
    "table_count",
    "tables_with_html",
    "tables_without_html",
    "formula_count",
    "formulas_with_latex",
    "formulas_without_latex",
    "cross_page_span_count",
    "affected_pages",
):
    if column in document_summary.columns:
        document_summary[column] = document_summary[column].fillna(0).astype(int)


document_summary

,filename,page_count,preproc_blocks,para_blocks,discarded_blocks,min_page_width,max_page_width,min_page_height,max_page_height,table_count,tables_with_html,tables_without_html,formula_count,formulas_with_latex,formulas_without_latex,cross_page_span_count,affected_pages
0,GDI_0685.pdf,9,88,88,11,595,595,841,841,4,4,0,0,0,0,0,0
1,GGH9.pdf,8,73,73,11,595,595,842,842,4,4,0,0,0,0,12,2
2,GGT_1247.pdf,10,121,121,14,595,595,841,841,10,9,1,8,8,0,2,1
3,GTL_0081.pdf,5,40,40,12,595,595,841,841,0,0,0,0,0,0,0,0
4,GTM10.pdf,9,86,86,13,595,595,842,842,2,2,0,0,0,0,2,1


## 26. Relatório textual rápido

In [50]:
total_documents = len(loaded_documents)

total_pages = int(pages_df["page_index"].count())

total_blocks = len(blocks_df)

total_lines = len(lines_df)

total_spans = len(spans_df)

total_tables = len(tables_df)

total_formulas = len(formulas_df)

total_cross_page = len(cross_page_spans)

total_anomalies = len(anomalies_df)


display(
    Markdown(
        f"""
## Resultado da inspeção

- Documentos carregados: **{total_documents}**
- Páginas inspecionadas: **{total_pages}**
- Blocos inventariados: **{total_blocks}**
- Linhas inventariadas: **{total_lines}**
- Spans inventariados: **{total_spans}**
- Tabelas identificadas: **{total_tables}**
- Fórmulas identificadas: **{total_formulas}**
- Spans com continuidade entre páginas: **{total_cross_page}**
- Anomalias estruturais registradas: **{total_anomalies}**
"""
    )
)


## Resultado da inspeção

- Documentos carregados: **5**
- Páginas inspecionadas: **41**
- Blocos inventariados: **877**
- Linhas inventariadas: **2475**
- Spans inventariados: **2627**
- Tabelas identificadas: **20**
- Fórmulas identificadas: **8**
- Spans com continuidade entre páginas: **16**
- Anomalias estruturais registradas: **6**


## 27. Exportação dos relatórios

In [51]:
EXPORTS: dict[str, pd.DataFrame] = {
    "validation.csv": validation_df,
    "middle_manifest.csv": middle_manifest,
    "documents.csv": document_summary,
    "pages.csv": pages_df,
    "blocks.csv": blocks_df,
    "block_type_counts.csv": (block_type_counts),
    "titles.csv": title_blocks,
    "lines.csv": lines_df,
    "spans.csv": spans_df,
    "span_type_counts.csv": (span_type_counts),
    "multi_line_blocks.csv": (multi_line_blocks),
    "multi_line_detail.csv": (multi_line_detail),
    "first_page_blocks.csv": (first_page_blocks),
    "first_page_lines.csv": (first_page_lines),
    "tables.csv": tables_df,
    "formulas.csv": formulas_df,
    "images.csv": images_df,
    "discarded_blocks.csv": (discarded_df),
    "noise_blocks.csv": noise_blocks,
    "noise_leaks.csv": noise_leaks,
    "cross_page_spans.csv": (cross_page_spans),
    "block_score_summary.csv": (block_score_summary),
    "low_score_blocks.csv": (low_score_blocks),
    "span_score_summary.csv": (span_score_summary),
    "block_keys.csv": block_keys_df,
    "block_key_summary.csv": (block_key_summary),
    "line_keys.csv": line_keys_df,
    "span_keys.csv": span_keys_df,
    "duplicate_summary.csv": (duplicate_summary),
    "anomalies.csv": anomalies_df,
    "anomaly_summary.csv": (anomaly_summary),
}


exported_paths: list[Path] = []

for filename, dataframe in EXPORTS.items():
    export_path = REPORTS_ROOT / filename

    dataframe.to_csv(
        export_path,
        index=False,
        encoding="utf-8-sig",
    )

    exported_paths.append(export_path)


pd.DataFrame({"exported_path": [str(path) for path in exported_paths]})

,exported_path
0,D:\baseia_v3\data\reports\middle_inspection\validation.csv
1,D:\baseia_v3\data\reports\middle_inspection\middle_manifest.csv
2,D:\baseia_v3\data\reports\middle_inspection\documents.csv
3,D:\baseia_v3\data\reports\middle_inspection\pages.csv
4,D:\baseia_v3\data\reports\middle_inspection\blocks.csv
5,D:\baseia_v3\data\reports\middle_inspection\block_type_counts.csv
6,D:\baseia_v3\data\reports\middle_inspection\titles.csv
7,D:\baseia_v3\data\reports\middle_inspection\lines.csv
8,D:\baseia_v3\data\reports\middle_inspection\spans.csv
9,D:\baseia_v3\data\reports\middle_inspection\span_type_counts.csv


## 28. Snapshot JSON do schema observado

Este arquivo registra apenas o inventário estrutural. Ele não representa
ainda o schema definitivo do IR.

In [52]:
schema_snapshot = {
    "documents": total_documents,
    "pages": total_pages,
    "blocks": total_blocks,
    "lines": total_lines,
    "spans": total_spans,
    "block_collections": sorted(
        blocks_df["block_collection"].dropna().unique().tolist()
    ),
    "block_types": sorted(
        str(value) for value in blocks_df["block_type"].dropna().unique().tolist()
    ),
    "span_types": sorted(
        str(value) for value in spans_df["span_type"].dropna().unique().tolist()
    ),
    "block_keys": sorted(block_keys_df["key"].dropna().unique().tolist()),
    "line_keys": sorted(line_keys_df["key"].dropna().unique().tolist()),
    "span_keys": sorted(span_keys_df["key"].dropna().unique().tolist()),
    "formula_types": sorted(FORMULA_TYPES),
    "image_types": sorted(IMAGE_TYPES),
    "noise_types": sorted(NOISE_TYPES),
}


SCHEMA_SNAPSHOT_PATH = REPORTS_ROOT / "observed_schema.json"


with SCHEMA_SNAPSHOT_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        schema_snapshot,
        file,
        ensure_ascii=False,
        indent=2,
    )


SCHEMA_SNAPSHOT_PATH

WindowsPath('D:/baseia_v3/data/reports/middle_inspection/observed_schema.json')

## 29. Critérios para a próxima etapa

A construção do IR poderá partir de `para_blocks`, preservando referências
para os elementos originais de `preproc_blocks`.

Antes disso, a próxima etapa deverá decidir explicitamente:

1. qual coleção de blocos será a fonte textual canônica;
2. como representar linhas e spans no IR;
3. como preservar `bbox`, página e ordem de leitura;
4. como representar tabelas, figuras e fórmulas;
5. como registrar blocos descartados sem misturá-los ao conteúdo;
6. como tratar blocos com continuidade entre páginas;
7. como representar candidatos a metadados sem classificá-los cedo demais.

In [53]:
print("Inspeção concluída.")
print(f"Relatórios: {REPORTS_ROOT}")
print(f"Schema observado: {SCHEMA_SNAPSHOT_PATH}")

Inspeção concluída.
Relatórios: D:\baseia_v3\data\reports\middle_inspection
Schema observado: D:\baseia_v3\data\reports\middle_inspection\observed_schema.json
